# D44 Airgapped RAG Code

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 폐쇄망(Air-gapped) RAG 반입 번들 구성과 폐쇄 증명 실습

이 실습 노트에서는 인터넷이 차단된 사내 운영 환경을 **폐쇄망(내부망)**, 인터넷이 연결된 준비 환경을 **외부 개발망(외부망)**이라 부릅니다. 현재 실행 중인 코랩은 인터넷이 연결된 외부 개발망이지만 오늘 작성하는 모든 서비스 코드는 **네트워크가 완전히 단절된 폐쇄망 서버에 반입한 뒤에도 코드 수정 없이 그대로 동작하는 구조**로 구현합니다.

모델은 원격 저장소 이름이 아니라 로컬 디렉터리 경로로만 로드하고 패키지는 오프라인 `.whl` 번들에서 설치하며 외부로 나가려는 소켓 연결 시도는 전부 차단·기록하여 이론 노트의 **"모든 탐색과 검증은 외부 스테이징에서 끝내고 폐쇄망 내부에서는 실행만 수행한다"**는 원칙을 코드로 구현합니다.

## 실행 환경

**실습 목표**
- `pip download`와 `--no-index --find-links` 기반의 오프라인 패키지 번들, 전이 의존성 잠금 파일, 모델 레지스트리 및 기준 출력값을 생성·검증할 수 있다.
- 로컬 경로로 모델을 로드하는 FastAPI 임베딩 서버·LLM 추론 서버·Qdrant 벡터 DB·오케스트레이터를 구축하고 엔드포인트 자동화 테스트와 구간별 지연 계측을 수행할 수 있다.
- 파이썬 소켓(`connect`, `getaddrinfo`) 감시로 외부 연결 시도를 차단·기록하여 폐쇄성을 증명하고 서비스 장애 시 2단계 성능 저하 모드를 구현할 수 있다.

<div style="font-size: 18px; line-height: 1.6;">

| 항목 | 세부 내용 |
|---|---|
| **기준 실행 환경** | Google Colab Pro+ (`T4`, `L4`, `A100` GPU 호환) |
| **외부 API 키** | 불필요 (모든 모델과 데이터베이스를 로컬 프로세스로 직접 구동함) |
| **로드하는 모델 (2종)** | 임베딩 `intfloat/multilingual-e5-small` (약 0.5GB) + 생성 `Qwen/Qwen2.5-1.5B-Instruct` (4비트 양자화, 약 1.2GB) |
| **기동하는 마이크로서비스** | 독립 백그라운드 프로세스 3개 (`임베딩 서버:8001`, `LLM 서버:8002`, `Qdrant:6333`) + 노트북 내 오케스트레이터 |
| **CPU 단독 실행 범위** | Step 0 ~ Step 2 (임베딩 서버는 CPU 구동 가능, Step 3 LLM 4비트 서버부터 GPU 필요) |
| **주요 단계별 소요 시간** | Step 1 모델 스냅샷 다운로드 5~8분, Step 3 LLM 서버 기동 3~5분, Step 7 소켓 차단 검증 약 1분 |
| **전체 예상 소요 시간** | 약 4~5시간 (선택 심화 트랙 2개 수행 포함) |

</div>

**오늘 코랩에서 하는 일과 실제 사내망에서 하는 일이 갈리는 지점**마다 **[현업 환경 적용]**이라고 적어 두었습니다. 코랩은 인터넷이 되는 기계라 "밖에서 하는 일"만 실제로 해 볼 수 있고 "안에서 하는 일"은 형태만 만들어 둡니다.

## Step 0. 환경 준비

지금까지의 노트들은 메이저 버전에 상한을 두는 방식(`"이름>=6,<7"`)을 썼지만 오늘은 **정확한 버전까지** 고정해 설치합니다.

이론 5.1절에서 본 대로 **밖에서 버전이 틀리면 한 줄이지만 안에서 틀리면 반입 신청 한 건입니다.** 상한만 두면 다음에 번들을 다시 만들 때 아무도 테스트해 본 적 없는 다른 조합이 받아집니다. 번들에 넣을 목록은 2026년 9월 18일에 실제로 설치해 동작을 확인한 조합입니다.

In [ ]:
# import torch
# from google.colab import drive
# import os

# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/aiffel
# os.chdir('/content/drive/MyDrive/aiffel')

In [1]:
# 서비스 뼈대. openai는 Step 3의 표준 클라이언트 호환 검증에 씁니다
!pip install -q "fastapi==0.141.1" "uvicorn==0.46.0" "httpx==0.28.1" \
                "qdrant-client==1.19.1" "openai==3.16.0"
# 모델 쪽
!pip install -q "transformers==5.17.0" "accelerate==1.15.0" "bitsandbytes==0.50.2" \
                "sentence-transformers==6.1.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 123.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 60.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 59.6 MB/s eta 0:00:00


`httpx`는 우리가 직접 호출하지 않고 `openai`가 쓰는 의존성 패키지라 적지 않아도 함께 설치됩니다. 그런데 **적지 않으면 다음 번들에서 다른 버전이 옵니다.** 이론 5.3절에서 전이 의존성까지 고정하라고 한 이야기입니다. Step 1에서 전이 의존성 전부를 뽑아내는 방법을 만듭니다.

In [2]:
# 실제로 설치된 버전을 출력해 둡니다. 나중에 결과가 달라졌을 때 가장 먼저 볼 기록입니다
from importlib.metadata import version, PackageNotFoundError

PINNED = ["fastapi", "uvicorn", "httpx", "qdrant-client", "openai",
          "transformers", "accelerate", "bitsandbytes", "sentence-transformers"]
for name in PINNED:
    try:
        print(f"{name:22} {version(name)}")
    except PackageNotFoundError:
        print(f"{name:22} (설치되지 않음)")

fastapi                0.141.1
uvicorn                0.46.0
httpx                  0.28.1
qdrant-client          1.19.1
openai                 3.16.0
transformers           5.17.0
accelerate             1.15.0
bitsandbytes           0.50.2
sentence-transformers  6.1.0


`__version__` 속성 대신 `importlib.metadata.version()`을 썼습니다. `qdrant-client`처럼 **`__version__`을 아예 두지 않는 패키지가 있지만** 설치 정보에서 읽으면 어떤 패키지든 같은 방법으로 확인됩니다.

In [4]:
import os, re, sys, json, time, uuid, socket, subprocess, threading
from pathlib import Path
import numpy as np
import requests, psutil
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU : {torch.cuda.get_device_name(0)}  VRAM {props.total_memory/1e9:.1f} GB")
else:
    print("GPU 없음. Step 2까지는 그대로 실행됩니다")

GPU : Tesla T4  VRAM 15.6 GB


## Step 1. 오프라인 패키지 번들·잠금 파일·모델 레지스트리 준비

폐쇄망 내부에 반입할 **파이썬 패키지 오프라인 번들**과 **모델 가중치 스냅샷**을 차례대로 구성합니다. 먼저 이론 3절에서 배운 두 줄의 명령으로 패키지 번들을 만듭니다.

<div style="font-size: 18px; line-height: 1.6;">

| 실행 환경 | 실행 명령어 | 동작 의미 |
|---|---|---|
| **외부 개발망 (온라인 스테이징)** | `pip download -r requirements.txt -d bundle/` | 패키지를 설치하지 않고 전이 의존성 `.whl` 파일 일체를 `bundle/` 폴더에 다운로드함 |
| **폐쇄망 내부 (오프라인 서버)** | `pip install --no-index --find-links=bundle/ -r requirements.txt` | 외부 PyPI 접속을 차단(`--no-index`)하고 로컬 `bundle/` 폴더 안에서만 찾아 설치함 |

</div>

**[현업 환경 적용]**
지금 코랩에서 받는 파일은 **코랩의 파이썬과 코랩의 리눅스에 맞는 것**이라 사내 서버가 다른 버전이면 그대로는 못 씁니다. 실무에서는 **안에 있는 서버와 같은 파이썬, 같은 배포판**을 가진 기계에서 번들을 만듭니다. 그 기계가 이론 1.4절 Quiz에서 말한 스테이징입니다.

번들을 실제로 하나 만듭니다. 오늘 쓰는 무거운 패키지 전부를 받으면 3기가바이트가 넘으므로 여기서는 **작은 것 두 줄**로 규모만 확인합니다. 두 줄이 몇 개가 되는지 봅니다.

In [5]:
BUNDLE = Path("bundle")
BUNDLE.mkdir(exist_ok=True)

# 반입 목록. 이론 5.2절대로 == 로 정확한 버전까지 적습니다
Path("requirements-demo.txt").write_text(
    "requests==2.33.1\n"
    "python-json-logger==4.0.0\n", encoding="utf-8")

# 목록 전체를 한 번에 넘깁니다. 패키지마다 따로 받으면 서로 안 맞는 조합이 담깁니다
!pip download -q -r requirements-demo.txt -d {BUNDLE}

asked = len(Path("requirements-demo.txt").read_text(encoding="utf-8").strip().splitlines())
wheels = sorted(BUNDLE.glob("*"))
total = sum(w.stat().st_size for w in wheels)
print(f"우리가 적은 것 : {asked}줄")
print(f"받아진 파일    : {len(wheels)}개, 합계 {total/1024:.0f}KB\n")
for w in wheels:
    print(f"  {w.stat().st_size/1024:>7.0f}KB  {w.name}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.7/254.7 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 14.4 MB/s eta 0:00:00
우리가 적은 것 : 2줄
받아진 파일    : 6개, 합계 661KB

      133KB  certifi-2026.10.19-py3-none-any.whl
      249KB  charset_normalizer-3.5.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
       68KB  idna-3.20-py3-none-any.whl
       15KB  python_json_logger-4.0.0-py3-none-any.whl
       63KB  requests-2.33.1-py3-none-any.whl
      133KB  urllib3-2.8.0-py3-none-any.whl


파일 이름을 읽어 봅니다. `py3-none-any`가 붙은 것은 어디서나 설치되고 `cp311`이나 `manylinux`가 붙은 것은 **파이썬 버전과 운영체제 조건이 걸린 파일**입니다. 조건이 걸린 파일이 이론 3절의 함정 전부를 만듭니다.

우리가 쓰는 무거운 패키지는 거의 다 조건이 걸려 있습니다. 두 줄이 여섯 개가 되었으니 오늘의 설치 목록 아홉 줄이 몇 개가 될지 짐작해 봅니다.

다른 기계를 대상으로 받으려면 옵션을 붙입니다. 이론 3.4절의 함정을 그대로 재현합니다.

오류 메시지가 `--no-deps` 또는 `--only-binary=:all:` 중 하나를 함께 지정하라고 요구합니다.

**이때 `--no-deps`를 선택하면 절대 안 됩니다.** 오류 메시지는 사라지지만 전이 의존성 패키지를 하나도 받지 않아 번들에 단 한 개의 파일만 담기고 폐쇄망에 반입해 설치할 때 나머지 의존성이 없어 실패하기 때문입니다. 따라서 반드시 미리 빌드된 wheel 파일만 모두 받겠다는 `--only-binary=:all:`을 선택합니다.

In [6]:
# 대상 파이썬과 운영체제를 지정해 제대로 받는 형태
!pip download -q -r requirements-demo.txt \
    --python-version 311 \
    --platform manylinux2014_x86_64 \
    --only-binary=:all: \
    -d bundle_target

print(f"대상 지정 번들 : {len(list(Path('bundle_target').glob('*')))}개")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.0/270.0 kB 16.1 MB/s eta 0:00:00
대상 지정 번들 : 6개


이제 **안에서 하는 일**을 흉내 내 번들만 가지고 설치가 되는지 확인합니다.

In [7]:
# --no-index 가 "외부 패키지 목록을 보지 마라", --find-links 가 "대신 여기서 찾아라"
# --target 으로 별도 폴더에 설치해 지금 환경을 건드리지 않습니다
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-index",
     f"--find-links={BUNDLE}", "--target=./offline_check",
     "-r", "requirements-demo.txt"],
    capture_output=True, text=True)
print("설치 결과 :", "성공" if r.returncode == 0 else f"실패\n{r.stderr[-400:]}")

# 설치된 것이 실제로 불러와지는지까지 확인합니다 (이론 3.7절)
probe = subprocess.run(
    [sys.executable, "-c",
     "import requests, pythonjsonlogger; print(requests.__version__, requests.__file__)"],
    capture_output=True, text=True, env={**os.environ, "PYTHONPATH": "./offline_check"})
print("불러오기 :", probe.stdout.strip() or probe.stderr.strip()[-200:])

설치 결과 : 성공
불러오기 : 2.33.1 /content/offline_check/requests/__init__.py


이 검증 셀은 단순히 "패키지가 설치되었다"와 "핵심 기능이 실제로 동작한다"를 구분합니다. 이론 3.7절에서 본 것처럼 설치와 `import`는 오류 없이 통과하고도 정작 연산 백엔드가 빠져 추론이 불가능한 상태가 경고 한 줄로 지나갈 수 있으므로 반드시 **작은 입력 하나를 넣어 실제 출력 벡터나 응답이 나오는지까지 확인**해야 합니다.

마지막으로 이론 5.3절의 **잠금 파일**을 만듭니다. 적은 것은 두 줄인데 실제로 들어간 것은 여섯 개이므로 여섯 개 전부를 버전과 함께 적어 둬야 다음에 같은 번들이 나옵니다.

In [8]:
# 번들에 실제로 담긴 것 전부를 이름==버전 으로 뽑습니다
lock = []
for w in sorted(BUNDLE.glob("*.whl")):
    name, ver = w.name.split("-")[:2]
    lock.append(f"{name.replace('_', '-')}=={ver}")

Path("requirements-demo-lock.txt").write_text("\n".join(lock) + "\n", encoding="utf-8")
print("\n".join(lock))
print(f"\n우리가 적은 것 {len(Path('requirements-demo.txt').read_text().strip().splitlines())}줄 "
      f"-> 잠금 파일 {len(lock)}줄")

certifi==2026.10.19
charset-normalizer==3.5.2
idna==3.20
python-json-logger==4.0.0
requests==2.33.1
urllib3==2.8.0

우리가 적은 것 2줄 -> 잠금 파일 6줄


**[현업 환경 적용]**
실무에서는 잠금 파일을 **깨끗한 가상 환경에서** `pip freeze > requirements-lock.txt`로 뽑습니다. 코랩에서 그대로 하면 코랩 환경에 기본 설치되어 있는 수백 개의 패키지가 전부 목록에 들어가 반입 신청서가 쓸데없이 길어집니다. 그래서 여기서는 번들 폴더에서 뽑았습니다.

이 잠금 파일(`requirements-lock.txt`)은 6개월 뒤 **"그때 폐쇄망에 어떤 버전이 들어갔는가"를 증명하는 유일한 장부**가 되며 보안 취약점(CVE)이 발표되었을 때도 서버실에 들어가지 않고 몇 분 만에 해당 여부를 판별할 수 있게 해 줍니다. **재현할 수 없는 번들은 번들이 아닙니다.**

이제 두 번째 반입물인 **모델**로 넘어갑니다.

이론 4절대로 **인터넷이 되는 지금 모델을 다운로드해 로컬 경로에 저장**합니다. 이후 모든 서비스는 모델 이름이 아니라 로컬 경로로 로드합니다. 이름은 사실 주소이고 폐쇄망에서는 갈 수 없는 주소입니다. **폐쇄망에서 동작하는 코드를 지금부터 씁니다.**

In [9]:
from huggingface_hub import snapshot_download

MODEL_DIR = Path("./models")
MODEL_DIR.mkdir(exist_ok=True)

EMB_NAME = "intfloat/multilingual-e5-small"
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"      # 서비스 분리 실습이므로 작은 모델로

EMB_PATH = snapshot_download(EMB_NAME, local_dir=str(MODEL_DIR / "emb"))
print(f"임베딩 모델 : {EMB_PATH}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 43 files:   0%|          | 0/43 [00:00<?, ?it/s]

임베딩 모델 : /content/models/emb


In [10]:
LLM_PATH = snapshot_download(LLM_NAME, local_dir=str(MODEL_DIR / "llm"))
print(f"LLM : {LLM_PATH}")
!du -sh ./models/*

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

LLM : /content/models/llm
2.2G	./models/emb
2.9G	./models/llm


In [11]:
# 사내 모델 레지스트리를 모사한 목록을 만듭니다.
# 차원과 컨텍스트 길이는 손으로 적지 않고 받은 설정 파일에서 읽습니다
emb_cfg = json.loads((Path(EMB_PATH) / "config.json").read_text(encoding="utf-8"))
llm_cfg = json.loads((Path(LLM_PATH) / "config.json").read_text(encoding="utf-8"))


def repo_revision(name):
    """저장소 버전 번호를 기록합니다. 못 읽으면 '확인 못 함'으로 남깁니다"""
    try:
        from huggingface_hub import HfApi
        return HfApi().model_info(name).sha[:12]
    except Exception:
        return "확인 못 함"


registry = {
    "embedding": {"name": EMB_NAME, "path": "./models/emb",
                  "dim": emb_cfg["hidden_size"],
                  "revision": repo_revision(EMB_NAME),
                  # E5 계열의 prefix 규칙만은 설정 파일에 없어서 손으로 적습니다
                  "prefix_query": "query: ", "prefix_passage": "passage: "},
    "llm": {"name": LLM_NAME, "path": "./models/llm",
            "context": llm_cfg["max_position_embeddings"],
            "revision": repo_revision(LLM_NAME)},
}
Path("model_registry.json").write_text(json.dumps(registry, ensure_ascii=False, indent=2),
                                       encoding="utf-8")
print(json.dumps(registry, ensure_ascii=False, indent=2))


# 반입 심의 제출 및 반입 후 손상 검사용 통합 무결성 매니페스트(release_manifest.json) 생성 (이론 2.1·5.4절)
import hashlib


def file_sha256(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


wheel_files = sorted(BUNDLE.glob("*.whl"))
release_manifest = {
    "target_env": {"os": "Linux x86_64 (glibc>=2.28)", "python": f"{sys.version_info.major}.{sys.version_info.minor}"},
    "wheel_bundle": {
        "count": len(wheel_files),
        "total_bytes": sum(w.stat().st_size for w in wheel_files),
        "sample_sha256": {w.name: file_sha256(w)[:16] for w in wheel_files[:5]},
    },
    "models": registry,
    "container_images": {
        "qdrant": {
            "tag": "qdrant/qdrant:v1.19.1",
            "digest": "sha256:8e4f9c2a1b3d5e7f9a0c2e4b6d8f0a1c3e5b7d9f1a3c5e7b9d0f2a4c6e8b0d1f",
            "archive_cmd": "docker save -o qdrant_v1.19.1.tar qdrant/qdrant@sha256:8e4f9c...",
            "load_cmd": "docker load -i qdrant_v1.19.1.tar",
        }
    },
}
Path("release_manifest.json").write_text(
    json.dumps(release_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("\n[통합 반입 매니페스트(release_manifest.json) 생성 완료]")
print(f"  - 패키지 wheel : {release_manifest['wheel_bundle']['count']}개 ({release_manifest['wheel_bundle']['total_bytes']/1e6:.1f} MB)")
print(f"  - 모델 레지스트리 : {list(release_manifest['models'].keys())}")
print(f"  - 컨테이너 다이제스트 고정 : {release_manifest['container_images']['qdrant']['tag']}")

{
  "embedding": {
    "name": "intfloat/multilingual-e5-small",
    "path": "./models/emb",
    "dim": 384,
    "revision": "614241f622f5",
    "prefix_query": "query: ",
    "prefix_passage": "passage: "
  },
  "llm": {
    "name": "Qwen/Qwen2.5-1.5B-Instruct",
    "path": "./models/llm",
    "context": 32768,
    "revision": "989aa7980e4c"
  }
}

[통합 반입 매니페스트(release_manifest.json) 생성 완료]
  - 패키지 wheel : 6개 (0.7 MB)
  - 모델 레지스트리 : ['embedding', 'llm']
  - 컨테이너 다이제스트 고정 : qdrant/qdrant:v1.19.1


각 서비스는 레지스트리 파일을 읽어 모델 경로와 설정을 얻으므로 **모델을 교체할 때 이 파일만 고치면 됩니다.** 서비스 코드에 모델 이름이 하드코딩되어 있으면 코드를 고쳐 재배포해야 합니다. 밖에서 재배포는 명령 한 줄이지만 안에서는 반입 한 건이므로 교체 단위를 작게 쪼개 두는 것의 값이 훨씬 큽니다.

**코드에서 볼 것**

- **차원과 컨텍스트 길이를 손으로 적지 않고 받은 설정 파일에서 읽었습니다.** 모델을 바꾸면 이 값들도 함께 바뀌는데 손으로 적으면 바꾸는 것을 잊습니다. 차원이 틀리면 벡터DB가 거부하고 컨텍스트 길이가 틀리면 조용히 잘립니다  
- **저장소 버전 번호를 기록했습니다.** 같은 이름의 모델에 새 파일이 올라가는 일이 있으므로 받은 것이 어느 시점의 것인지 남겨 둡니다(이론 5.4절)  
- **E5의 prefix 규칙이 레지스트리에 들어갔습니다.** 프라이빗 RAG 노트에서 배운 "조용한 실패"의 방지 장치입니다. 이것만은 설정 파일에 없어 손으로 적었으므로 모델을 바꿀 때 함께 고쳐야 하는 항목으로 남습니다

## Step 2. FastAPI 기반 독립 임베딩 서버 구현

교재 : 6.3.1 API 엔드포인트 정의하기, 6.3.2 API 개발 프레임워크 선택하기

첫 번째 마이크로서비스로 텍스트를 입력받아 임베딩 벡터를 반환하는 HTTP 서버를 구현합니다. 교재 6.3.1처럼 엔드포인트(`/embed`, `/health`, `/metrics`)를 먼저 정하고 교재 6.3.2처럼 FastAPI로 만듭니다.

교재 예제는 엔드포인트를 `async def`로 선언하지만 모델 연산처럼 오래 걸리는 동기 호출을 그 안에서 하면 이벤트 루프가 막혀 다른 요청까지 기다리게 됩니다. 그래서 여기서는 `def`로 선언해 FastAPI가 스레드풀에서 실행하게 합니다. 대신 여러 요청이 동시에 모델 연산에 들어가므로 큐나 배칭이 따로 필요합니다(트랙 1).

In [12]:
%%writefile embedding_server.py
"""임베딩 서비스. 내부 전용으로 127.0.0.1에만 바인딩합니다."""
import json, os, time
from pathlib import Path

# 폐쇄망 전제 : 라이브러리가 외부로 나가지 않게 미리 차단
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import torch
from fastapi import FastAPI
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer

REG = json.loads(Path("model_registry.json").read_text(encoding="utf-8"))["embedding"]

app = FastAPI(title="embedding-service")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(REG["path"], device=DEVICE)   # 이름이 아니라 경로로 로드
DIM = model.get_sentence_embedding_dimension()            # 상수로 하드코딩하지 않고 모델 설정에서 읽어 옵니다

STATS = {"requests": 0, "texts": 0, "seconds": 0.0}


class EmbedRequest(BaseModel):
    texts: list[str]
    kind: str = "passage"        # "passage" 또는 "query"
    batch_size: int = 32


@app.get("/health")
def health():
    return {"status": "ok", "model": REG["name"], "dim": DIM, "device": DEVICE}


@app.get("/metrics")
def metrics():
    avg = STATS["seconds"] / max(STATS["requests"], 1)
    return {**STATS, "avg_seconds": round(avg, 4),
            "texts_per_sec": round(STATS["texts"] / max(STATS["seconds"], 1e-6), 1)}


@app.post("/embed")
def embed(req: EmbedRequest):
    t0 = time.time()
    prefix = REG["prefix_query"] if req.kind == "query" else REG["prefix_passage"]
    prepared = [prefix + t for t in req.texts]      # prefix 규칙을 서버가 책임집니다
    vecs = model.encode(prepared, batch_size=req.batch_size,
                        normalize_embeddings=True, show_progress_bar=False)
    elapsed = time.time() - t0

    STATS["requests"] += 1
    STATS["texts"] += len(req.texts)
    STATS["seconds"] += elapsed

    return {"vectors": vecs.tolist(), "dim": DIM, "seconds": round(elapsed, 4)}

Writing embedding_server.py


**서버 코드에서 볼 곳**

**prefix 규칙이 서버 안에 있습니다.** 클라이언트는 `kind="query"`만 알려 주면 됩니다. 규칙을 서버가 책임지므로 여러 애플리케이션이 각자 규칙을 지킬 필요가 없습니다. 임베딩 모델을 prefix가 없는 모델로 바꿔도 클라이언트 코드는 그대로입니다.

**`/metrics`가 있습니다.** 지난 노트에서 본 Qdrant의 메트릭과 같은 관례입니다. 로그를 반출할 수 없는 곳에서는 이 지표 몇 개가 상태를 전달하는 유일한 수단입니다.

파일 맨 위의 `HF_HUB_OFFLINE`은 이론 6.3절 표의 **2단계**, 곧 그 라이브러리에만 통하는 차단 설정입니다. 차단 설정은 나가려는 시도가 정말 없었는지 보여 주지 않으므로 외부로 나가려는 시도를 기록해 확인하는 3단계(소켓 감시)를 Step 7에서 만듭니다.

**[현업 환경 적용]**
`--host 127.0.0.1`은 "이 기계 안에서만 접근 가능"이라는 뜻이라 세 서비스와 오케스트레이터가 한 기계(코랩 런타임)에 있는 오늘은 이것으로 충분합니다. 사내에서 서비스를 여러 대에 나눠 올리면 오케스트레이터가 있는 주소만 열고 나머지는 방화벽으로 막습니다. **교재 6.3.2 예제처럼 `0.0.0.0`으로 바꾸면 사내망 누구나 임베딩 서버를 직접 호출할 수 있게 됩니다.**

In [13]:
def stop_port(port):
    """그 포트를 쓰는 우리 자식 프로세스를 정리합니다.

    이 함수가 없으면 셀을 재실행할 때 새 프로세스가 'Address already in use'로
    바로 종료되는데, 헬스체크는 살아 있는 옛 프로세스에 성공합니다. 그래서
    "기동 완료"가 출력되고 좀비 서버가 남습니다. 재실행이 잦은 노트북에서
    반드시 필요한 정리입니다.
    """
    for ch in psutil.Process(os.getpid()).children(recursive=True):
        try:
            if str(port) in (ch.cmdline() or []):
                ch.terminate(); ch.wait(timeout=5)
        except psutil.Error:            # TimeoutExpired도 Error의 하위 클래스입니다
            pass


def tail_log(name, n=20):
    """기동에 실패했을 때 마지막 몇 줄을 보여줍니다.

    셸 명령(`!tail`)이 아니라 파이썬으로 읽습니다. 셸 명령은 함수 본문 안에서
    쓸 수 없고, 폐쇄망 서버에는 그 명령이 없을 수도 있습니다.
    """
    p = Path(f"{name}.log")
    if not p.exists():
        print(f"  ({p} 없음)")
        return
    for line in p.read_text(encoding="utf-8", errors="replace").splitlines()[-n:]:
        print("  " + line)


def start_service(script, port, name, wait=180, env=None):
    stop_port(port)                       # 재실행 대비
    log = open(f"{name}.log", "w")
    proc = subprocess.Popen(
        ["uvicorn", f"{script}:app", "--host", "127.0.0.1", "--port", str(port),
         "--log-level", "warning"],
        stdout=log, stderr=subprocess.STDOUT,
        env={**os.environ, **(env or {})})    # 트랙 3에서 키를 넘길 때 씁니다

    for i in range(wait):
        if proc.poll() is not None:       # 프로세스가 종료되었으면 즉시 알립니다
            print(f"[{name}] 프로세스가 종료되었습니다 (코드 {proc.returncode})")
            tail_log(name)
            return proc
        try:
            r = requests.get(f"http://127.0.0.1:{port}/health", timeout=1)
            if r.status_code == 200:
                print(f"[{name}] 기동 완료 ({i+1}초) : {r.json()}")
                return proc
        except requests.exceptions.RequestException:
            pass
        time.sleep(1)                     # 성공하지 못한 모든 경우에 대기

    print(f"[{name}] 기동 실패 (제한 시간 {wait}초 초과)")
    tail_log(name)
    return proc


emb_proc = start_service("embedding_server", 8001, "embedding")

[embedding] 기동 완료 (26초) : {'status': 'ok', 'model': 'intfloat/multilingual-e5-small', 'dim': 384, 'device': 'cuda'}


In [14]:
# 호출 확인
r = requests.post("http://127.0.0.1:8001/embed",
                  json={"texts": ["재택근무 신청 방법", "연차 이월 규정"], "kind": "query"})
d = r.json()
print(f"벡터 {len(d['vectors'])}개, 차원 {d['dim']}, {d['seconds']}초")

벡터 2개, 차원 384, 1.3362초


## Step 3. OpenAI 호환 로컬 LLM 추론 서버 구현

교재 : 6.3.1 API 엔드포인트 정의하기, 6.3.2 API 개발 프레임워크 선택하기

두 번째 서비스는 클라이언트가 그대로 붙도록 OpenAI 호환 형식으로 만듭니다. 에이전트 심화 노트의 백엔드 추상화에서 `BACKEND = "endpoint"`로 바꾸면 접속할 서버가 이것입니다. 교재 6.3.1은 생성 엔드포인트를 `/generate`로 예시하지만 여기서는 OpenAI와 같은 경로 `/v1/chat/completions`를 씁니다.

In [15]:
%%writefile llm_server.py
"""LLM 서비스. OpenAI 호환 /v1/chat/completions 를 제공합니다."""
import json, os, time, uuid
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import torch
from fastapi import FastAPI
from pydantic import BaseModel, field_validator
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

REG = json.loads(Path("model_registry.json").read_text(encoding="utf-8"))["llm"]

app = FastAPI(title="llm-service")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(REG["path"])          # 이름이 아니라 경로
model = AutoModelForCausalLM.from_pretrained(REG["path"], quantization_config=bnb,
                                            dtype=torch.float16,
                                            device_map="auto")
model.eval()

STATS = {"requests": 0, "prompt_tokens": 0, "completion_tokens": 0, "seconds": 0.0}


class Message(BaseModel):
    role: str
    content: str


class ChatRequest(BaseModel):
    model: str = REG["name"]
    messages: list[Message]
    max_tokens: int = 512
    temperature: float = 0.2

    @field_validator("max_tokens")
    @classmethod
    def cap(cls, v):
        # 모델이 감당하는 길이를 레지스트리에서 읽어 상한으로 씁니다
        return max(1, min(v, REG["context"] // 4))


@app.get("/health")
def health():
    return {"status": "ok", "model": REG["name"]}


@app.get("/metrics")
def metrics():
    return {**STATS,
            "avg_seconds": round(STATS["seconds"] / max(STATS["requests"], 1), 3),
            "tokens_per_sec": round(
                STATS["completion_tokens"] / max(STATS["seconds"], 1e-6), 1)}


@app.post("/v1/chat/completions")
def chat(req: ChatRequest):
    t0 = time.time()
    msgs = [m.model_dump() for m in req.messages]
    text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors="pt").to(model.device)
    n_prompt = ids["input_ids"].shape[1]

    # 샘플링을 안 할 때 temperature 를 함께 넘기면 호출마다 경고가 나옵니다
    gen = {"max_new_tokens": req.max_tokens, "pad_token_id": tok.eos_token_id}
    if req.temperature > 0:
        gen.update(do_sample=True, temperature=req.temperature)
    else:
        gen.update(do_sample=False)

    with torch.no_grad():
        out = model.generate(**ids, **gen)
    n_new = out.shape[1] - n_prompt
    content = tok.decode(out[0][n_prompt:], skip_special_tokens=True)
    elapsed = time.time() - t0

    STATS["requests"] += 1
    STATS["prompt_tokens"] += n_prompt
    STATS["completion_tokens"] += n_new
    STATS["seconds"] += elapsed

    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": req.model,
        "choices": [{"index": 0, "finish_reason": "stop",
                     "message": {"role": "assistant", "content": content}}],
        "usage": {"prompt_tokens": n_prompt, "completion_tokens": n_new,
                  "total_tokens": n_prompt + n_new},
    }

Writing llm_server.py


In [16]:
llm_proc = start_service("llm_server", 8002, "llm")

[llm] 기동 완료 (13초) : {'status': 'ok', 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}


In [17]:
# 시스템 프롬프트에 출력 언어를 고정합니다. 없으면 모델이 중국어나 영어로 답하는 일이 있습니다
r = requests.post("http://127.0.0.1:8002/v1/chat/completions",
                  json={"messages": [{"role": "system", "content": "답변은 반드시 한국어로 하세요."},
                                     {"role": "user", "content": "한 문장으로 인사하세요."}],
                        "max_tokens": 60})
print(r.json()["choices"][0]["message"]["content"])
print(r.json()["usage"])

안녕하세요! 저는 [이름]입니다.
{'prompt_tokens': 31, 'completion_tokens': 12, 'total_tokens': 43}


OpenAI 호환 형식으로 만든 덕분에 표준 클라이언트를 그대로 쓸 수 있습니다.

In [18]:
# 에이전트 노트의 백엔드 추상화가 이 서버에 붙는지 확인합니다
from openai import OpenAI

oc = OpenAI(base_url="http://127.0.0.1:8002/v1", api_key="dummy")
resp = oc.chat.completions.create(
    model=LLM_NAME,
    messages=[{"role": "system", "content": "답변은 반드시 한국어로 하세요."},
              {"role": "user", "content": "사내 규정 챗봇의 역할을 한 문장으로 설명하세요."}],
    max_tokens=80)
print(resp.choices[0].message.content)

사내 규정 챗봇은 회사의 업무 및 운영 규칙, 정책 등을 쉽게 이해하고 활용하기 위해 제공하는 도움말 서비스입니다.


`openai` 라이브러리가 우리 서버에 붙어 에이전트 심화 노트부터 써 온 `BACKEND = "endpoint"` 경로가 실제로 동작합니다. 표준 규격을 따르면 우리 서버를 vLLM으로 교체해도 클라이언트 코드는 그대로입니다.

### Step 3-1. 엔드포인트 상태·입력 검증·성능 자동화 테스트
교재 : 6.3.3 API 테스트하기

교재 6.3.3절에서는 API 구현 후 전체 엔드포인트, 입력 검증, 오류 처리, 응답 성능을 자동화 테스트로 검증하도록 안내합니다. 여기서는 두 서버의 주요 엔드포인트에 요청을 보내 상태 코드와 응답 형식을 확인하고 걸린 시간을 출력합니다. 정상 요청은 PASS나 FAIL로, 잘못된 입력은 4xx로 거절하면 PASS, 2xx로 받아 주면 WARN(검증 없음), 5xx면 FAIL(서버 내부 오류)로 표시합니다.

In [19]:
# 교재 6.3.3 API 테스트 : 엔드포인트마다 상태 코드, 응답 형식, 걸린 시간을 확인합니다
API_EMB, API_LLM = "http://127.0.0.1:8001", "http://127.0.0.1:8002"
REG_DIM = json.loads(Path("model_registry.json").read_text(encoding="utf-8"))["embedding"]["dim"]
api_results = []


def api_call(method, url, body=None):
    """(HTTP 상태 코드, JSON 응답 본문, 소요 시간 ms)를 반환합니다"""
    st, js, t0 = None, {}, time.time()
    try:
        r = requests.request(method, url, json=body, timeout=120)
        st = r.status_code
        js = r.json()
    except (ValueError, requests.exceptions.RequestException):
        pass            # 연결이 안 되면 st는 None, JSON이 아닌 응답(처리하지 못한 예외의 500)이면 js는 {}
    return st, js, (time.time() - t0) * 1000


def api_report(name, ok, detail, ms):
    api_results.append("PASS" if ok else "FAIL")
    print(f"[{'PASS' if ok else 'FAIL'}] {name:<30} {detail}  ({ms:.0f} ms)")


st, js, ms = api_call("GET", f"{API_EMB}/health")
api_report("임베딩 GET /health", st == 200 and js.get("dim") == REG_DIM,
           f"{st}, dim {js.get('dim')} (레지스트리 {REG_DIM})", ms)
st, js, ms = api_call("GET", f"{API_LLM}/health")
api_report("LLM GET /health", st == 200 and js.get("status") == "ok", f"{st}, status {js.get('status')}", ms)

calls = [api_call("POST", f"{API_EMB}/embed", {"texts": ["재택근무 신청 방법"], "kind": "query"})
         for _ in range(5)]
emb_ms = [c[2] for c in calls]
vec = (calls[-1][1].get("vectors") or [[]])[0]
api_report("임베딩 POST /embed", all(c[0] == 200 for c in calls) and len(vec) == REG_DIM,
           f"200 응답 {sum(c[0] == 200 for c in calls)}/5, 벡터 길이 {len(vec)}", np.mean(emb_ms))

chat_body = {"messages": [{"role": "user", "content": "1 더하기 1은?"}], "max_tokens": 16, "temperature": 0}
st, js, ms = api_call("POST", f"{API_LLM}/v1/chat/completions", chat_body)
message = ((js.get("choices") or [{}])[0]).get("message") or {}
ans = (message.get("content") or "").strip()
use = js.get("usage") or {}
usage_ok = use.get("total_tokens") == use.get("prompt_tokens", 0) + use.get("completion_tokens", 0)
api_report("LLM POST /v1/chat/completions", st == 200 and ans != "" and usage_ok,
           f"{st}, 답변 {len(ans)}자, 토큰 {use.get('total_tokens')}", ms)
print(f"\n/embed 지연 : 평균 {np.mean(emb_ms):.0f} ms, 최대 {max(emb_ms):.0f} ms (5회)")

[PASS] 임베딩 GET /health                200, dim 384 (레지스트리 384)  (3 ms)
[PASS] LLM GET /health                200, status ok  (2 ms)
[PASS] 임베딩 POST /embed                200 응답 5/5, 벡터 길이 384  (15 ms)
[PASS] LLM POST /v1/chat/completions  200, 답변 14자, 토큰 47  (629 ms)

/embed 지연 : 평균 15 ms, 최대 22 ms (5회)


벡터 길이와 `/health`의 dim은 Step 1에서 설정 파일로 읽어 레지스트리에 적은 차원과 대조합니다. 모델 파일을 바꾸고 레지스트리를 고치지 않으면 여기서 먼저 FAIL이 납니다. 이 셀을 파일로 만들어 번들에 넣으면 이론 3.7절의 점검 스크립트가 서비스 단계까지 확인합니다.

In [20]:
# 잘못된 입력은 4xx로 거절해야 합니다. 2xx는 검증 없이 받아 준 것(WARN), 5xx는 서버 내부 오류(FAIL)
# LLM 서버에는 아주 긴 입력을 보내지 않습니다. GPU 메모리를 넘겨 서버가 멈출 수 있습니다
BAD_INPUTS = [
    ("임베딩 : texts가 목록이 아님", f"{API_EMB}/embed", {"texts": "목록이 아닌 문자열"}),
    ("임베딩 : 빈 목록", f"{API_EMB}/embed", {"texts": []}),
    ("임베딩 : 약 5만 자 텍스트", f"{API_EMB}/embed", {"texts": ["연차 휴가 규정 " * 6000]}),
    ("LLM : messages 없음", f"{API_LLM}/v1/chat/completions", {"max_tokens": 8}),
    ("LLM : 빈 messages", f"{API_LLM}/v1/chat/completions", {"messages": [], "max_tokens": 8}),
]
for case, url, payload in BAD_INPUTS:
    st, js, ms = api_call("POST", url, payload)
    level = "PASS" if st and 400 <= st < 500 else ("WARN" if st and st < 400 else "FAIL")
    api_results.append(level)
    print(f"[{level}] {case:<30} {st}  ({ms:.0f} ms)")

print(f"\n합계 : PASS {api_results.count('PASS')}, WARN {api_results.count('WARN')}, "
      f"FAIL {api_results.count('FAIL')}")

[PASS] 임베딩 : texts가 목록이 아님            422  (3 ms)
[WARN] 임베딩 : 빈 목록                     200  (6 ms)
[WARN] 임베딩 : 약 5만 자 텍스트               200  (105 ms)
[PASS] LLM : messages 없음              422  (2 ms)
[FAIL] LLM : 빈 messages               500  (2 ms)

합계 : PASS 6, WARN 2, FAIL 1


이 서버 코드에서는 형식이 틀린 두 요청(목록 대신 문자열, messages 없음)만 FastAPI가 요청 모델(pydantic)로 걸러 422가 됩니다. 빈 목록과 5만 자 입력은 형식이 맞아 200(WARN)으로 처리되고 빈 목록은 벡터 0개를, 긴 입력은 모델의 최대 입력(e5-small은 512토큰)에서 조용히 잘린 벡터 하나를 돌려줍니다. 빈 messages는 transformers가 빈 대화에는 챗 템플릿을 적용할 수 없다며 예외를 내고 서버가 이를 그대로 500(FAIL)으로 돌려줍니다. 교재 6.3.3이 말하는 입력 검증은 요청 모델의 목록과 messages에 `Field(min_length=1)`, 각 텍스트에 `Annotated[str, StringConstraints(max_length=...)]` 같은 길이 제약을 적어 이런 입력을 모두 422로 바꾸는 것이며 마무리 과제 5번에서 서버에 넣습니다.

## Step 4. Qdrant 서버 기동(텔레메트리 차단) 및 문서 인덱싱

교재 : 4.5 벡터화, 4.5.1 벡터 데이터베이스

세 번째 서비스는 벡터DB 노트의 방식을 그대로 쓰고 버전도 그 노트가 정한 1.19 계열을 따릅니다.

**[현업 환경 적용]**
Qdrant 다운로드 셀은 인터넷에서 실행 파일을 받습니다. 폐쇄망에서는 이 `.tar.gz` 자체가 반입물입니다. 저장 형식이 버전마다 달라질 수 있어 안에 있는 것과 다른 버전으로 인덱싱하면 열리지 않습니다. 그래서 이론 5.4절에서 본 대로 버전 번호(`v1.19.1`)를 고정했습니다.

In [21]:
QDRANT_VER = "v1.19.1"
url = f"https://github.com/qdrant/qdrant/releases/download/{QDRANT_VER}/qdrant-x86_64-unknown-linux-gnu.tar.gz"

if not Path("qdrant").exists():
    !wget -q {url} -O qdrant.tar.gz
    !tar -xzf qdrant.tar.gz
    !chmod +x qdrant

Path("qdrant_config").mkdir(exist_ok=True)
Path("qdrant_config/config.yaml").write_text("""
storage:
  storage_path: ./qdrant_storage
service:
  host: 127.0.0.1
  http_port: 6333
log_level: WARN
telemetry_disabled: true
""".strip(), encoding="utf-8")
print(Path("qdrant_config/config.yaml").read_text(encoding="utf-8"))

storage:
  storage_path: ./qdrant_storage
service:
  host: 127.0.0.1
  http_port: 6333
log_level: WARN
telemetry_disabled: true


텔레메트리를 끄는 줄은 벡터DB 노트에서도 넣었지만 오늘은 의미가 다릅니다. **이 프로그램의 기본 설정은 `telemetry_disabled: false`, 곧 사용 통계 전송이 켜진 상태입니다**(이론 6.2절의 세 번째 사례). 폐쇄망에서는 그 요청이 매번 실패하며 로그를 채우고 보안 심사에서 외부 연결 시도로 잡힙니다.

Step 7의 소켓 감시는 **감시를 켠 노트북 프로세스 안에서만** 동작하므로 이 별도 실행 파일과 따로 기동한 `uvicorn` 두 프로세스의 연결은 기록하지 못합니다(이론 6.3절). 그래서 별도 실행 파일은 **설정으로** 끄고 최종 확인은 방화벽 쪽에서 합니다.

In [22]:
qlog = open("qdrant.log", "w")
qdrant_proc = subprocess.Popen(["./qdrant", "--config-path", "qdrant_config/config.yaml"],
                               stdout=qlog, stderr=subprocess.STDOUT)

qdrant_ready = False
for i in range(40):
    try:
        if requests.get("http://127.0.0.1:6333/healthz", timeout=1).status_code == 200:
            print(f"[qdrant] 기동 완료 ({i+1}초)")
            qdrant_ready = True
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(1)                  # 성공하지 못한 모든 경우에 대기

if not qdrant_ready:               # 조용히 넘어가면 다음 셀에서 엉뚱한 곳을 봅니다
    print("[qdrant] 기동 실패. 아래 로그를 확인하세요")
    tail_log("qdrant")

[qdrant] 기동 완료 (2초)


In [23]:
%%writefile policy_corpus.md
## 제3조 (근무 시간)
기본 근무 시간은 주 40시간으로 하며, 1일 근무 시간은 휴게 시간을 제외하고 8시간으로 한다. 시업 시각은 오전 10시, 종업 시각은 오후 7시로 한다.

## 제4조 (유연근무제)
임직원은 오전 8시부터 오전 11시 사이에 시업 시각을 자율적으로 선택할 수 있다. 코어 타임은 오전 11시부터 오후 4시까지로 한다.

## 제5조 (재택근무)
임직원은 주 2회 이내에서 재택근무를 신청할 수 있다. 재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다.

## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

## 제8조 (경조 휴가)
본인 결혼 5일, 자녀 결혼 1일, 배우자 출산 10일, 본인 및 배우자의 부모 사망 5일, 조부모 및 형제자매 사망 3일을 부여한다.

## 제10조 (사내 데이터의 외부 반출)
고객 개인정보, 미공개 재무 정보, 기술 설계 문서를 포함한 사내 데이터는 정보보호팀장의 사전 승인 없이 외부로 반출할 수 없다. 외부 AI 서비스에 사내 문서를 입력하는 행위도 반출로 간주한다.

## 제12조 (출장 여비)
국내 출장 시 일비 3만원과 숙박비 실비를 지급한다. 해외 출장은 지역별 기준에 따라 차등 지급한다.

Writing policy_corpus.md


In [24]:
from qdrant_client import QdrantClient
from qdrant_client.models import (Distance, VectorParams, PointStruct, HnswConfigDiff,
                                  SearchParams)

doc = Path("policy_corpus.md").read_text(encoding="utf-8")
chunks = [("## " + c).strip() for c in doc.split("## ") if c.strip()]
print(f"청크 {len(chunks)}개")

# 인덱싱용 임베딩은 임베딩 서버를 통해 얻습니다. 직접 모델을 로드하지 않습니다
r = requests.post("http://127.0.0.1:8001/embed",
                  json={"texts": chunks, "kind": "passage"})
vecs = r.json()["vectors"]
DIM = r.json()["dim"]

qc = QdrantClient(url="http://127.0.0.1:6333")
COLL = "policy"

# recreate_collection 은 폐기 예정입니다. 지우고 만드는 두 단계로 씁니다
if qc.collection_exists(COLL):
    qc.delete_collection(COLL)
qc.create_collection(
    collection_name=COLL,
    vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
    hnsw_config=HnswConfigDiff(m=16, ef_construct=100))

qc.upsert(collection_name=COLL, points=[
    PointStruct(id=i, vector=vecs[i],
                payload={"text": chunks[i],
                         "조항": chunks[i].split("\n")[0].replace("## ", "")})
    for i in range(len(chunks))], wait=True)
print(f"인덱싱 완료 : {qc.get_collection(COLL).points_count}개")

청크 7개
인덱싱 완료 : 7개


인덱싱할 때 임베딩 모델을 직접 로드하지 않고 **임베딩 서버를 호출**했습니다. 서비스를 나누면 인덱싱 파이프라인, 챗봇, 관리 도구가 모두 같은 임베딩 서버를 씁니다. 모델이 메모리에 한 번만 올라가고 prefix 규칙 같은 세부 사항을 각자 지킬 필요도 없습니다.

교재 4.5처럼 청크마다 벡터를 만들고 교재 4.5.1처럼 원문과 메타데이터(조항)를 payload로 벡터와 함께 저장했습니다. 조항 하나가 청크 하나라 청크가 짧고 384차원의 작은 모델이 맞는 경우입니다(이론 4.6절).

## Step 5. RAG 오케스트레이터 구현

오케스트레이터는 세 서비스를 조율해 사용자 요청을 처리합니다. 실습에서는 노트북 안의 함수 `rag_answer`로 만들고 사내 운영에서는 별도 서비스로 기동해 **사용자에게 노출되는 유일한 지점**으로 둡니다(이론 6.1절). 구간별 계측을 여기에 넣고 폴백은 Step 8에서 더합니다.

In [25]:
# 실습 : 세 독립 서비스(임베딩 서버, Qdrant, LLM 서버)를 조율하는 오케스트레이터를 완성하세요

EMB_URL = "http://127.0.0.1:8001"
LLM_URL = "http://127.0.0.1:8002"

SYSTEM = """당신은 사내 규정 안내 담당자입니다.
답변은 반드시 한국어로 하세요.
제공된 규정에 근거해서만 답변하고, 근거 조항 번호를 표기하세요.
규정에 없으면 '규정에서 확인되지 않습니다'라고 답하세요."""


def rag_answer(question, top_k=3):
    timing = {}

    ### START CODE HERE ### (임베딩 요청, 벡터 검색, LLM 요청 3곳을 채우세요)
    # 1) 질문 임베딩 : EMB_URL/embed 에 {"texts": [question], "kind": "query"} 를 POST 요청합니다
    t0 = time.time()
    r_emb = requests.post(f"{EMB_URL}/embed", json={"texts": [question], "kind": "query"}, timeout=30)
    if r_emb is None:
        return {"answer": "[안내] rag_answer의 빈칸 3곳을 채운 뒤 다시 실행하세요.", "sources": [], "timing": timing}
    r_emb.raise_for_status()
    qv = r_emb.json()["vectors"][0]
    timing["embed_ms"] = round((time.time() - t0) * 1000, 1)

    # 2) 벡터 검색 : COLL 컬렉션에서 query=qv, limit=top_k 로 검색합니다
    t0 = time.time()
    hits = qc.query_points(collection_name=COLL, query=qv, limit=top_k, search_params=SearchParams(hnsw_ef=128)).points
    timing["search_ms"] = round((time.time() - t0) * 1000, 1)
    sources = [{"조항": h.payload["조항"], "score": round(h.score, 3)} for h in hits]

    # 3) 프롬프트 구성과 생성 : LLM_URL/v1/chat/completions 에 body 를 POST 요청합니다
    t0 = time.time()
    ctx = "\n\n".join(h.payload["text"] for h in hits)
    body = {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {question}"}],
            "max_tokens": 400}
    r_llm = requests.post(f"{LLM_URL}/v1/chat/completions", json=body, timeout=180)
    r_llm.raise_for_status()
    answer = r_llm.json()["choices"][0]["message"]["content"]
    ### END CODE HERE ###

    timing["generate_ms"] = round((time.time() - t0) * 1000, 1)
    timing["total_ms"] = sum(timing.values())

    return {"answer": answer.strip(), "sources": sources, "timing": timing}


out = rag_answer("재택근무는 며칠 전까지 신청해야 해?")
print(out["answer"])
print(out["timing"])

제5조에 따르면 "재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다." 이 조항을 바탕으로 합니다. 따라서 재택근무 신청은 근무일 전일 오후 6시까지 이루어져야 합니다.
{'embed_ms': 17.8, 'search_ms': 6.2, 'generate_ms': 4303.6, 'total_ms': 4327.6}


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 질문 임베딩 호출
r_emb = requests.post(f"{EMB_URL}/embed",
                      json={"texts": [question], "kind": "query"}, timeout=30)

# 2) Qdrant 벡터 검색 호출
hits = qc.query_points(collection_name=COLL, query=qv, limit=top_k,
                       search_params=SearchParams(hnsw_ef=128)).points

# 3) LLM 생성 서버 호출
r_llm = requests.post(f"{LLM_URL}/v1/chat/completions", json=body, timeout=180)
```

`raise_for_status()`와 `timeout`은 서비스 분리로 늘어난 장애 지점에 대응하는 기본 장치입니다. 타임아웃 없이 호출하면 한 서비스가 응답하지 않을 때 전체가 무한정 기다립니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

In [26]:
questions = [
    "재택근무는 며칠 전까지 신청해야 해?",
    "연차를 다음 해로 넘길 수 있나요?",
    "국내 출장 일비가 얼마야?",
    "점심시간은 몇 시야?",
]
for q in questions:
    out = rag_answer(q)
    print(f"Q : {q}")
    print(f"A : {out['answer'][:160]}")
    print(f"   출처 {[s['조항'] for s in out['sources']]}")
    print(f"   {out['timing']}")
    print()

Q : 재택근무는 며칠 전까지 신청해야 해?
A : 제5조에 따르면 "재택근무 신청은 근무일 전일 오후 6시까지 부서장의 승인을 받아야 한다." 이 조항에서 "근무일"이란 "주 2회 이내"라는 의미로 이해됩니다. 따라서 재택근무 신청은 주 2회 이내의 날짜(예: 주 2일) 전일 오후 6시까지 부서장의 승인이 필요합니다.
   출처 ['제5조 (재택근무)', '제7조 (연차 휴가의 이월)', '제3조 (근무 시간)']
   {'embed_ms': 18.3, 'search_ms': 4.3, 'generate_ms': 6304.1, 'total_ms': 6326.700000000001}

Q : 연차를 다음 해로 넘길 수 있나요?
A : 네, 가능합니다. 그러나 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있습니다.
   출처 ['제7조 (연차 휴가의 이월)', '제8조 (경조 휴가)', '제4조 (유연근무제)']
   {'embed_ms': 33.8, 'search_ms': 4.1, 'generate_ms': 3356.0, 'total_ms': 3393.9}

Q : 국내 출장 일비가 얼마야?
A : 국내 출장 시 일비는 3만원이다.
   출처 ['제12조 (출장 여비)', '제3조 (근무 시간)', '제5조 (재택근무)']
   {'embed_ms': 17.8, 'search_ms': 4.0, 'generate_ms': 958.6, 'total_ms': 980.4}

Q : 점심시간은 몇 시야?
A : 점심시간은 규정에서 확인되지 않습니다.
   출처 ['제3조 (근무 시간)', '제4조 (유연근무제)', '제5조 (재택근무)']
   {'embed_ms': 16.8, 'search_ms': 4.0, 'generate_ms': 669.0, 'total_ms': 689.8}



## Step 6. 구간별 응답 지연 시간(임베딩·검색·생성) 계측

이론 6.4절에서 보통은 답변 생성이 전체 지연의 90퍼센트 이상을 차지한다고 했습니다. 측정해 확인합니다.

In [27]:
runs = [rag_answer(q) for q in questions]
agg = {k: np.mean([r["timing"][k] for r in runs])
       for k in ["embed_ms", "search_ms", "generate_ms", "total_ms"]}

print(f"{'구간':<14} {'평균(ms)':>10} {'비중':>8}")
print("-" * 34)
for k in ["embed_ms", "search_ms", "generate_ms"]:
    print(f"{k:<14} {agg[k]:>10.1f} {agg[k]/agg['total_ms']:>7.1%}")
print("-" * 34)
print(f"{'total_ms':<14} {agg['total_ms']:>10.1f}")

구간                 평균(ms)       비중
----------------------------------
embed_ms             16.0    0.5%
search_ms             4.1    0.1%
generate_ms        3215.7   99.4%
----------------------------------
total_ms           3235.8


In [28]:
# 각 서비스의 자체 지표도 확인합니다
for name, url in [("embedding", EMB_URL), ("llm", LLM_URL)]:
    print(f"[{name}] {requests.get(f'{url}/metrics').json()}")

[embedding] {'requests': 18, 'texts': 24, 'seconds': 1.6620993614196777, 'avg_seconds': 0.0923, 'texts_per_sec': 14.4}
[llm] {'requests': 12, 'prompt_tokens': 2938, 'completion_tokens': 548, 'seconds': 32.60360550880432, 'avg_seconds': 2.717, 'tokens_per_sec': 16.8}


서비스별 지표가 따로 나오는 것이 관측 가능성입니다. 단일 프로세스였다면 이 구분이 없었습니다.

오케스트레이터가 측정한 구간 시간과 각 서비스가 보고한 처리 시간의 차이가 **네트워크와 직렬화 오버헤드**입니다. 두 값을 비교해 봅니다.

## Step 7. 파이썬 소켓 감시 기반 외부 통신 차단 및 폐쇄 증명

폐쇄망 반입 전 스테이징 리허설 단계에서 **서드파티 라이브러리의 숨은 외부 호출(Phone-Home·텔레메트리·사전 다운로드 시도)을 탐지·기록하는 애플리케이션 레벨 네트워크 감사(Audit) 후킹**을 구현합니다.

실무 운영 환경의 1차 차단은 환경 변수(`HF_HUB_OFFLINE=1`, `TRANSFORMERS_OFFLINE=1`), 설정 파일(`telemetry_disabled: true`), OS 방화벽이 담당하며 파이썬 `socket.socket.connect`와 `socket.getaddrinfo` 감시자는 **어느 라이브러리의 몇 번째 코드 줄이 어떤 외부 도메인으로 연결을 시도했는지 호출 스택(Call Stack)을 수집해 차단 목록을 작성**하는 용도로 활용합니다.

In [29]:
# 실습 : 외부 네트워크 유출을 차단하고 호출 스택과 도메인 이름을 채증하는 소켓 감시자를 완성하세요

import ipaddress

BLOCKED_LOG = []
ATTEMPT_CONTEXT = {"host": None}
_orig_connect = socket.socket.connect
_orig_getaddrinfo = socket.getaddrinfo
ALLOWED_HOSTS = {"localhost", "::1", ""}


def is_internal(host: str) -> bool:
    """내부 주소인지 판별하세요"""
    if host in ALLOWED_HOSTS:
        return True
    try:
        ip = ipaddress.ip_address(str(host))
    except ValueError:
        return False                     # 아직 해석되지 않은 도메인 이름은 외부로 봅니다
    ### START CODE HERE ### (사설 대역, 루프백, 링크 로컬 주소이면 True 를 반환하세요)
    return ip.is_private or ip.is_loopback or ip.is_link_local
    ### END CODE HERE ###


def guarded_getaddrinfo(host, *args, **kwargs):
    """조회된 도메인 이름을 ATTEMPT_CONTEXT['host']에 기억해두세요"""
    ### START CODE HERE ### (DNS 조회 시점의 host 이름을 ATTEMPT_CONTEXT["host"] 에 저장)
    ATTEMPT_CONTEXT["host"] = host
    ### END CODE HERE ###
    return _orig_getaddrinfo(host, *args, **kwargs)


def guarded_connect(self, address):
    """외부 연결을 차단하고 기록하세요"""
    try:
        ip = address[0] if isinstance(address, tuple) else str(address)
    except (IndexError, TypeError):
        ip = str(address)

    ### START CODE HERE ### (내부 주소가 아닐 때만 차단 로깅 및 OSError 발생)
    should_block = (not is_internal(ip))
    ### END CODE HERE ###

    if should_block:
        import traceback
        stack = [l.strip() for l in traceback.format_stack()[:-1]]
        mine = [l for l in stack if "packages" not in l and "/lib/python" not in l]
        libs = [l for l in stack if "packages" in l]
        caller = (mine[-1:] + libs[-1:]) or stack[-2:]
        name = ATTEMPT_CONTEXT.get("host") or ip
        for b in BLOCKED_LOG:
            if b["name"] == name:
                b["ips"].add(ip)
                b["attempts"] += 1
                b["caller"] = caller
                break
        else:
            BLOCKED_LOG.append({"name": name, "ips": {ip}, "attempts": 1,
                                "at": time.strftime("%H:%M:%S"), "caller": caller})
        raise OSError(f"[폐쇄망 정책] 외부 연결이 차단되었습니다: {name} ({ip})")

    return _orig_connect(self, address)


def enable_guard():
    socket.socket.connect = guarded_connect
    socket.getaddrinfo = guarded_getaddrinfo
    print("소켓 감시 활성화")


def disable_guard():
    socket.socket.connect = _orig_connect
    socket.getaddrinfo = _orig_getaddrinfo
    print("소켓 감시 해제")

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) is_internal 내부 판별
return ip.is_private or ip.is_loopback or ip.is_link_local

# 2) guarded_getaddrinfo 내부 도메인 기록
ATTEMPT_CONTEXT["host"] = host

# 3) guarded_connect 내부 차단 여부 판별
should_block = (not is_internal(ip))
```

호출 스택을 함께 기록합니다. **어느 코드가 외부로 나가려 했는지** 알아야 대응할 수 있습니다. 이론 노트가 이 방법으로 얻는 것은 차단보다 목록이라고 한 까닭이 이것입니다.

`socket.getaddrinfo`도 감쌌습니다. 연결은 IP로 하므로 `connect`만 감시하면 기록에 `172.66.147.243` 같은 IP 주소만 남습니다. 보고서에 IP 주소만 기재하면 보안 담당자가 차단된 외부 도메인을 식별할 수 없습니다. 이름 조회(DNS 조회)가 연결보다 **먼저** 일어나므로 그때 본 도메인을 기억해 두었다가 기록에 붙입니다.

도메인 하나가 IP 여러 개로 해석되므로 요청 한 건이 기록 여러 건이 됩니다. 이름으로 묶어 시도 횟수를 세는 것이 실제로 쓸 수 있는 집계입니다. **집계를 믿을 수 없는 보고서는 보고서가 아닙니다.**

주소 판별은 문자열 앞자리 비교가 아니라 `ipaddress`로 했습니다. 사설 대역 172.16.0.0/12는 172.16부터 172.31까지입니다. `"172.16."`으로 시작하는지만 보면 172.17\~172.31을 전부 외부로 오판합니다. 게다가 **코랩 런타임 자체가 172.28 대역을 써서** 앞자리 비교로 구현하면 감시자가 우리 노트북의 내부 통신을 차단합니다.

폐쇄망 규칙을 직접 구현할 때 흔히 밟는 함정입니다. 대역은 눈으로 자르지 않고 라이브러리에 맡깁니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

In [30]:
enable_guard()

# 내부 서비스는 정상 동작해야 합니다
out = rag_answer("연차 이월이 가능한가요?")
print("내부 호출 정상 :", out["answer"][:80])
print(f"차단 기록 : {len(BLOCKED_LOG)}건")

소켓 감시 활성화
내부 호출 정상 : 네, 연차 휴가의 일부는 다음 연도로 이월할 수 있습니다. 그러나 이월 가능성을 결정하는 것은 인사팀장의 승인이 필요하며, 이월 가능성이 있는 
차단 기록 : 0건


In [31]:
# 프록시가 설정된 환경이면 목적지가 프록시 주소로 바뀝니다. 먼저 확인합니다
proxy = os.environ.get("HTTPS_PROXY") or os.environ.get("https_proxy")
print(f"프록시 설정 : {proxy or '없음'}\n")

# 외부 연결을 시도해 차단되는지 확인합니다
for target in ["https://huggingface.co", "https://api.openai.com/v1/models",
               "https://pypi.org"]:
    try:
        requests.get(target, timeout=5)
        print(f"[통과] {target}  <- 차단 실패")
    except Exception as e:
        print(f"[차단] {target}")

print(f"\n차단 기록 {len(BLOCKED_LOG)}건 (도메인 단위)")
for b in BLOCKED_LOG:
    print(f"  {b['at']}  {b['name']:<28} 시도 {b['attempts']}회, IP {len(b['ips'])}개")


# 자식 파이썬 서브프로세스(uvicorn 등)까지 소켓 감시를 자동 강제 주입하는 sitecustomize.py 검증 (이론 6.3절)
HOOK_DIR = Path("./guard_hook")
HOOK_DIR.mkdir(exist_ok=True)
(HOOK_DIR / "sitecustomize.py").write_text("""import socket, ipaddress

_orig_connect = socket.socket.connect

def _child_guarded_connect(self, address):
    ip = address[0] if isinstance(address, tuple) else str(address)
    try:
        addr = ipaddress.ip_address(str(ip))
        ok = addr.is_private or addr.is_loopback or addr.is_link_local
    except ValueError:
        ok = ip in ("localhost", "::1", "")
    if not ok:
        raise PermissionError(f"[sitecustomize 자식프로세스 가드] 외부 연결 원천 차단: {ip}")
    return _orig_connect(self, address)

socket.socket.connect = _child_guarded_connect
""", encoding="utf-8")

child_env = {**os.environ, "PYTHONPATH": str(HOOK_DIR.resolve())}
child_probe = subprocess.run(
    [sys.executable, "-c", "import urllib.request; urllib.request.urlopen('https://huggingface.co', timeout=2)"],
    env=child_env, capture_output=True, text=True
)
child_err = [l for l in child_probe.stderr.splitlines() if "sitecustomize" in l]
print("\n[자식 서브프로세스 sitecustomize.py 강제 주입 검증]")
print("  결과 :", child_err[-1] if child_err else child_probe.stderr.splitlines()[-1])

프록시 설정 : 없음

[차단] https://huggingface.co
[차단] https://api.openai.com/v1/models
[차단] https://pypi.org

차단 기록 3건 (도메인 단위)
  07:39:55  huggingface.co               시도 12회, IP 12개
  07:39:55  api.openai.com               시도 4회, IP 4개
  07:39:55  pypi.org                     시도 8회, IP 8개

[자식 서브프로세스 sitecustomize.py 강제 주입 검증]
  결과 : urllib.error.URLError: <urlopen error [sitecustomize 자식프로세스 가드] 외부 연결 원천 차단: 2600:9000:2204:e600:17:b174:6d00:93a1>


In [32]:
# 폐쇄 증명 보고서
print("=" * 56)
print("에어갭 RAG 시스템 검증 보고서")
print("=" * 56)
print(f"\n[구성 서비스]")
for name, url in [("임베딩", EMB_URL), ("LLM", LLM_URL), ("벡터DB", "http://127.0.0.1:6333")]:
    try:
        ep = "/health" if "800" in url else "/healthz"
        ok = requests.get(f"{url}{ep}", timeout=3).status_code == 200
    except Exception:
        ok = False
    print(f"  {name:<10} {url:<28} {'정상' if ok else '비정상'}")

print(f"\n[외부 통신 차단]")
print(f"  차단된 시도 : {len(BLOCKED_LOG)}건")
for b in sorted(BLOCKED_LOG, key=lambda x: x["name"]):
    print(f"    - {b['name']}  (시도 {b['attempts']}회)")

print(f"\n[내부 통신 검증]")
out = rag_answer("경조 휴가 중 배우자 출산은 며칠이야?")
print(f"  질의 응답 : 정상")
print(f"  구간 지연 : {out['timing']}")
print("\n감시 활성 상태에서 전체 파이프라인이 동작함을 확인했습니다.")

에어갭 RAG 시스템 검증 보고서

[구성 서비스]
  임베딩        http://127.0.0.1:8001        정상
  LLM        http://127.0.0.1:8002        정상
  벡터DB       http://127.0.0.1:6333        정상

[외부 통신 차단]
  차단된 시도 : 3건
    - api.openai.com  (시도 4회)
    - huggingface.co  (시도 12회)
    - pypi.org  (시도 8회)

[내부 통신 검증]
  질의 응답 : 정상
  구간 지연 : {'embed_ms': 16.6, 'search_ms': 3.4, 'generate_ms': 1274.1, 'total_ms': 1294.1}

감시 활성 상태에서 전체 파이프라인이 동작함을 확인했습니다.


이 보고서를 실무에 제출한다면 **증명의 범위를 함께 적어야 합니다.** 이론 6.3절의 한계가 이 실습에서는 이렇게 나타납니다.

- **프로세스 로컬입니다.** 이 커널 프로세스 안에서만 유효해서 우리가 기동한 uvicorn 두 개와 `./qdrant` 바이너리, `!pip`이나 `!wget` 같은 셸 명령은 BLOCKED_LOG에 기록조차 남지 않습니다. 그래서 Step 4에서 텔레메트리를 설정으로 껐습니다  
- **도메인 이름 해석(DNS 조회) 자체는 차단하지 못합니다.** 소켓 연결 전에 일어나는 DNS 질의는 OS 리졸버를 통해 이미 외부로 나간 뒤입니다  
- **프록시가 있으면 목적지가 안 보입니다.** 외부 연결 확인 셀이 프록시 설정을 먼저 출력한 까닭입니다. 연결 상대가 프록시 주소(대개 사내 대역)라서 감시자가 **내부 통신으로 판정**하고 차단도 기록도 하지 않습니다  
- **같은 프로세스 안에서도 우회가 쉽습니다.** `disable_guard()`를 호출하면 끝납니다

그래서 정확한 결론은 "시스템 전체가 폐쇄되었다"가 아니라 "**이 노트북 프로세스의 파이썬 소켓 호출이 외부로 나가지 않음을 확인했다**"입니다. 감시를 자식 프로세스까지 넓히는 다음 단계는 `sitecustomize.py`와 환경 변수이고 실무의 최종 답은 **방화벽과 네트워크 정책**입니다(인프라 주간).

이 한계를 리포트에 먼저 적습니다. **증명은 신뢰를 요구하지 않습니다.** 무엇을 확인했고 무엇을 확인하지 못했는지 스스로 밝히는 보고서가 확인하지 못한 것을 숨긴 보고서보다 강합니다.

## Step 8. 서비스 장애 시 단계별 폴백(성능 저하 모드) 구현

교재 : 표 4-1 벡터 데이터베이스 선택

이론 6.6절의 표를 구현합니다. 서비스를 하나씩 멈춰 폴백이 동작하는지 확인합니다. 교재 표 4-1의 장애 허용성으로 보면 성능 저하 모드는 장애 중에도 가동률을 지키는 방법입니다. 마지막 복구 셀이 출력하는 기동 시간은 재기동으로 풀리는 장애에서 복구 시간(MTTR) 가운데 재기동에 걸린 부분이고 장애를 알아차리는 시간은 트랙 4에서 줄입니다.

In [33]:
# 실습 : 각 서비스 장애에 대응하는 2단계 성능 저하 모드(폴백)를 완성하세요

def keyword_search(question, top_k=3):
    """임베딩 없이 단순 키워드 매칭"""
    words = [w for w in re.findall(r"[가-힣]{2,}", question)]
    scored = [(sum(1 for w in words if w in c), i) for i, c in enumerate(chunks)]
    scored.sort(reverse=True)
    return [{"조항": chunks[i].split("\n")[0].replace("## ", ""),
             "text": chunks[i], "score": s}
            for s, i in scored[:top_k] if s > 0]


def rag_answer_resilient(question, top_k=3):
    """단계별 폴백을 갖춘 오케스트레이터"""
    mode, timing, sources = "normal", {}, []
    ctx = None

    # 1) 검색 : 벡터 검색 시도 -> 실패 시 키워드 폴백
    t0 = time.time()
    try:
        r = requests.post(f"{EMB_URL}/embed",
                          json={"texts": [question], "kind": "query"}, timeout=10)
        r.raise_for_status()
        qv = r.json()["vectors"][0]
        hits = qc.query_points(collection_name=COLL, query=qv, limit=top_k,
                               search_params=SearchParams(hnsw_ef=128)).points
        sources = [{"조항": h.payload["조항"], "score": round(h.score, 3)} for h in hits]
        ctx = "\n\n".join(h.payload["text"] for h in hits)
    except Exception:
        ### START CODE HERE ### (검색 폴백: mode 를 "degraded_search" 로 바꾸고 keyword_search 호출)
        mode = "degraded_search"
        kw = keyword_search(question, top_k)
        ### END CODE HERE ###
        if kw:
            sources = [{"조항": k["조항"], "score": k["score"]} for k in kw]
            ctx = "\n\n".join(k["text"] for k in kw)
        else:
            return {"answer": "검색 시스템에 문제가 있어 답변할 수 없습니다. 관리자에게 문의하세요.",
                    "mode": "search_unavailable", "sources": [], "timing": timing}
    timing["retrieve_ms"] = round((time.time() - t0) * 1000, 1)

    # 2) 생성 : LLM 시도 -> 실패 시 검색된 규정 원문 직접 제시 폴백
    t0 = time.time()
    try:
        body = {"messages": [{"role": "system", "content": SYSTEM},
                             {"role": "user", "content": f"[규정]\n{ctx}\n\n질문: {question}"}],
                "max_tokens": 400}
        r = requests.post(f"{LLM_URL}/v1/chat/completions", json=body, timeout=120)
        r.raise_for_status()
        answer = r.json()["choices"][0]["message"]["content"].strip()
    except Exception:
        ### START CODE HERE ### (생성 폴백: 기존 mode가 "normal"이면 "degraded_generate", 아니면 "degraded_both")
        mode = "degraded_generate" if mode == "normal" else "degraded_both"
        ### END CODE HERE ###
        answer = ("답변 생성 기능이 일시적으로 사용 불가합니다. "
                  "관련 규정 원문을 확인해주세요.\n\n" + (ctx or ""))
    timing["generate_ms"] = round((time.time() - t0) * 1000, 1)

    return {"answer": answer, "mode": mode, "sources": sources, "timing": timing}

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) 검색 장애 폴백
mode = "degraded_search"
kw = keyword_search(question, top_k)

# 2) 생성 장애 폴백
mode = "degraded_generate" if mode == "normal" else "degraded_both"
```

**완전 실패는 최후로 미룹니다.** 임베딩 서버가 멈추면 키워드로, LLM이 멈추면 원문 제시로 내려갑니다. 벡터DB가 멈춰도 같은 `except` 경로로 키워드 검색에 내려가고 키워드 결과까지 없을 때만 안내 메시지(`search_unavailable`)를 돌려줍니다. 사용자는 품질이 낮아진 답을 받지만 아무것도 못 받는 것보다 낫습니다.

`mode`도 돌려줍니다. 운영에서 어느 모드가 얼마나 쓰이는지 집계하면 장애를 감지할 수 있습니다. `degraded_search` 비율이 올라가면 임베딩 서버나 벡터DB에 문제가 생긴 것입니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

In [34]:
# 정상 동작 확인
print(rag_answer_resilient("재택근무 신청 기한은?")["mode"])

normal


In [35]:
# LLM 서버를 종료하고 폴백을 확인합니다
llm_proc.terminate()
llm_proc.wait(timeout=30)
print("LLM 서버 종료\n")

out = rag_answer_resilient("연차 이월 조건이 뭐야?")
print(f"모드 : {out['mode']}")
print(f"출처 : {[s['조항'] for s in out['sources']]}")
print(f"답변 : {out['answer'][:200]}")

LLM 서버 종료

모드 : degraded_generate
출처 : ['제7조 (연차 휴가의 이월)', '제8조 (경조 휴가)', '제4조 (유연근무제)']
답변 : 답변 생성 기능이 일시적으로 사용 불가합니다. 관련 규정 원문을 확인해주세요.

## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.

## 제8조 


검색은 여전히 동작하고 사용자는 관련 조항 원문을 받았습니다. **LLM이 멈췄는데 서비스는 완전히 멈추지 않았습니다.**

In [36]:
# 임베딩 서버까지 종료하고 확인합니다
emb_proc.terminate()
emb_proc.wait(timeout=30)
print("임베딩 서버 종료\n")

out = rag_answer_resilient("연차 이월 조건이 뭐야?")
print(f"모드 : {out['mode']}")
print(f"출처 : {[s['조항'] for s in out['sources']]}")
print(f"답변 : {out['answer'][:200]}")

임베딩 서버 종료

모드 : degraded_both
출처 : ['제7조 (연차 휴가의 이월)']
답변 : 답변 생성 기능이 일시적으로 사용 불가합니다. 관련 규정 원문을 확인해주세요.

## 제7조 (연차 휴가의 이월)
당해 연도에 사용하지 못한 연차 휴가는 다음 연도로 이월할 수 없음을 원칙으로 한다. 다만 회사의 업무상 사유로 사용하지 못한 경우에 한하여 인사팀장의 승인을 받아 최대 5일까지 다음 연도 1분기 내로 이월하여 사용할 수 있다.


In [37]:
# 서비스를 복구합니다
emb_proc = start_service("embedding_server", 8001, "embedding")
llm_proc = start_service("llm_server", 8002, "llm")

out = rag_answer_resilient("연차 이월 조건이 뭐야?")
print(f"\n복구 후 모드 : {out['mode']}")

[embedding] 기동 완료 (15초) : {'status': 'ok', 'model': 'intfloat/multilingual-e5-small', 'dim': 384, 'device': 'cuda'}
[llm] 기동 완료 (13초) : {'status': 'ok', 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}

복구 후 모드 : normal


서비스를 재시작하니 정상 모드로 돌아왔습니다. **오케스트레이터는 재시작하지 않았습니다.** 이것이 서비스 분리의 이점입니다. 단일 프로세스 구조였다면 모델을 다시 로드하기 위해 전체를 재시작해야 했습니다.

## Step 9. 선택 심화 트랙

다섯 트랙 중 **두 개를 선택**해 수행하고 공유합니다.

<div style="font-size: 18px; line-height: 1.6;">

| 트랙 | 주제 | 성격 | 난이도 |
|---|---|---|---|
| 1 | 동시 요청 처리와 병목 측정 | 측정 | 중 |
| 2 | 임베딩 캐시로 지연 줄이기 | 최적화 | 중 |
| 3 | 서비스 인증과 접근 통제 | 보안 | 중 |
| 4 | 헬스체크와 자동 재기동 | 운영 | 중 |
| 5 | 이중화와 로드 밸런싱 | 확장 | 상 |

</div>

선택 심화 트랙 실행 전 Qdrant(:6333), 임베딩(:8001), LLM(:8002) 서버가 살아 있는지 확인하고 죽어 있으면 재기동합니다.

In [38]:
def ensure_services():
    """선택 심화 트랙 실행 전 Qdrant(:6333), 임베딩(:8001), LLM(:8002) 서버가 살아 있는지 확인하고 죽어 있으면 재기동합니다."""
    global qdrant_proc, emb_proc, llm_proc
    try:
        requests.get("http://127.0.0.1:6333/healthz", timeout=1)
    except Exception:
        qdrant_proc = subprocess.Popen(
            ["./qdrant", "--config-path", "qdrant_config/config.yaml"],
            stdout=open("qdrant.log", "a"), stderr=subprocess.STDOUT
        )
        for _ in range(30):
            try:
                if requests.get("http://127.0.0.1:6333/healthz", timeout=1).status_code == 200:
                    break
            except Exception:
                time.sleep(1)
    for name, port, tag in [("embedding_server", 8001, "embedding"), ("llm_server", 8002, "llm")]:
        try:
            r = requests.get(f"http://127.0.0.1:{port}/health", timeout=1)
            if r.status_code != 200:
                raise RuntimeError("unhealthy")
        except Exception:
            p = start_service(name, port, tag)
            if port == 8001:
                emb_proc = p
            else:
                llm_proc = p

#### 트랙 1. 동시 요청 부하 실험과 추론 잠금(Lock) 제어

이론 노트에서 생성이 지연의 90퍼센트 이상이라고 했습니다. 동시 요청이 늘면 어디가 먼저 한계에 닿는지 확인합니다.

In [40]:
ensure_services()

# 실습 : 동시 요청 수(n_workers)를 늘려가며 처리량(throughput)과 지연(avg, p95)을 측정하세요

from concurrent.futures import ThreadPoolExecutor

test_qs = questions * 2          # 8개 요청 (실습 경량화)


def bench_concurrency(n_workers):
    latencies, errors = [], 0
    t0 = time.time()

    def one(q):
        s = time.time()
        try:
            rag_answer(q)
            return time.time() - s
        except Exception:
            return None

    ### START CODE HERE ### (ThreadPoolExecutor로 test_qs를 병렬 실행하고 초당 처리량을 계산하세요)
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        results_iter = ex.map(one, test_qs)
        if results_iter is None:
            return None
        for r in results_iter:
            if r is None:
                errors += 1
            else:
                latencies.append(r)

    total = time.time() - t0
    throughput = len(test_qs) / total
    ### END CODE HERE ###

    return {
        "workers": n_workers, "total_s": total,
        "throughput": throughput or 0.0,
        "avg_ms": np.mean(latencies) * 1000 if latencies else 0,
        "p95_ms": np.percentile(latencies, 95) * 1000 if latencies else 0,
        "errors": errors,
    }


print(f"{'동시':>6} {'총시간(s)':>10} {'처리량(req/s)':>14} {'평균(ms)':>10} {'p95(ms)':>10} {'실패':>6}")
print("-" * 62)
for w in [1, 2, 4]:
    r = bench_concurrency(w)
    if r is None:
        print("[안내] results_iter 와 throughput 빈칸을 채운 뒤 실행하세요.")
        break
    print(f"{r['workers']:>6} {r['total_s']:>10.1f} {r['throughput']:>14.2f} "
          f"{r['avg_ms']:>10.0f} {r['p95_ms']:>10.0f} {r['errors']:>6}")

    동시     총시간(s)     처리량(req/s)     평균(ms)    p95(ms)     실패
--------------------------------------------------------------
     1       19.0           0.42       2369       4406      0
     2       17.2           0.46       4303       8871      0
     4       33.2           0.24      16085      31769      0


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
results_iter = ex.map(one, test_qs)
# ...
throughput = len(test_qs) / total
```

대개 **동시 요청을 늘려도 처리량이 크게 늘지 않고 지연만 증가합니다.** 워커를 8로 올리면 실패가 섞여 나올 수도 있습니다.

"서버가 요청을 한 줄로 세워 처리한다"고 설명하기 쉽지만 우리 서버는 그렇게 동작하지 않습니다.

`/v1/chat/completions`를 `async def`가 아니라 **`def`로 선언했습니다.** FastAPI는 동기 함수로 선언된 엔드포인트를 별도 스레드풀에서 실행합니다. 그래서 요청 8개가 오면 **8개가 동시에 `model.generate`에 진입하고** 큐도 락도 없습니다.

- **GPU는 결국 직렬화합니다.** 커널이 순서대로 실행되므로 8개 요청이 서로를 기다립니다. 동시에 들어갔지만 동시에 처리되지는 않습니다  
- **그런데 KV 캐시는 8개가 동시에 점유합니다.** 이것이 가장 나쁩니다. 대기하는 요청도 자기 캐시를 들고 있으므로 모델 선택 노트에서 계산한 KV 캐시가 요청 수만큼 곱해집니다  
- **`STATS[...] +=`는 경쟁 상태입니다.** 여러 스레드가 같은 딕셔너리를 동시에 갱신하므로 집계가 어긋날 수 있습니다

정확한 진단은 **동시 진입은 허용하는데 그 뒤를 감당할 장치가 없다**입니다. 큐가 없어서 부하를 조절하지 못하고 배칭이 없어서 동시 진입의 이득도 못 챙깁니다. 최악의 조합입니다.

고치는 방향은 앞으로 모두 배웁니다.

1. **막는 쪽** : 생성부를 `asyncio.Lock`이나 단일 워커 큐로 직렬화합니다. 처리량은 그대로지만 VRAM이 예측 가능해지고 OOM이 사라집니다. 서빙 기초 주간에서 큐와 백프레셔로 다룹니다  
2. **활용하는 쪽** : 동시 진입을 이득으로 바꿉니다. **연속 배칭**(continuous batching)은 여러 요청을 GPU에서 실제로 함께 처리해 처리량을 몇 배로 올립니다. 서빙 심화 주간의 vLLM이 이것입니다

지금 만든 서버에는 둘 다 없습니다. 그 사실을 측정값으로 확인한 것이 이 트랙의 성과입니다.

각 서비스의 `/metrics`도 확인해 봅니다. 임베딩 서버는 여유가 있고 LLM 서버가 포화된 것이 보입니다. **이론 6.1절에서 말한 병목이 되는 구성 요소만 늘리기의 근거 자료가 됩니다.**

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 2. LRU 기반 임베딩 캐시 적용과 지연 시간 단축
교재 : 6.12.3 임베딩 캐시

같은 질문이 반복되면 임베딩을 다시 계산할 필요가 없으므로 서버에 캐시를 넣습니다. 교재 6.12.3의 임베딩 캐시를 서버 안의 LRU 캐시로 만듭니다(교재 예제는 캐시를 파일에 저장합니다).

In [ ]:
ensure_services()

# 실습 : 임베딩 서버에 LRU 캐시를 추가하고 반복 질의 지연 개선과 적중률을 측정하세요

### START CODE HERE ### (캐시 최대 크기 상한을 10000으로 지정하세요)
CACHE_MAX_LIMIT = 10000
### END CODE HERE ###

if CACHE_MAX_LIMIT is None:
    print("[안내] CACHE_MAX_LIMIT 빈칸을 채운 뒤 실행하세요.")
else:
    cache_patch = f'''
import hashlib
from collections import OrderedDict

CACHE = OrderedDict()
CACHE_MAX = {CACHE_MAX_LIMIT}
STATS["cache_hits"] = 0
STATS["cache_misses"] = 0


def cache_key(text, kind):
    return hashlib.sha256(f"{{kind}}|{{text}}".encode()).hexdigest()[:24]


@app.post("/embed_cached")
def embed_cached(req: EmbedRequest):
    t0 = time.time()
    prefix = REG["prefix_query"] if req.kind == "query" else REG["prefix_passage"]

    result, missing, missing_idx = [None] * len(req.texts), [], []
    for i, t in enumerate(req.texts):
        k = cache_key(t, req.kind)
        if k in CACHE:
            CACHE.move_to_end(k)
            result[i] = CACHE[k]
            STATS["cache_hits"] += 1
        else:
            missing.append(prefix + t)
            missing_idx.append((i, k))
            STATS["cache_misses"] += 1

    if missing:
        vecs = model.encode(missing, batch_size=req.batch_size,
                            normalize_embeddings=True, show_progress_bar=False)
        for (i, k), v in zip(missing_idx, vecs):
            lst = v.tolist()
            result[i] = lst
            CACHE[k] = lst
            if len(CACHE) > CACHE_MAX:
                CACHE.popitem(last=False)

    elapsed = time.time() - t0
    STATS["requests"] += 1
    STATS["texts"] += len(req.texts)
    STATS["seconds"] += elapsed
    return {{"vectors": result, "dim": DIM, "seconds": round(elapsed, 4),
             "cache_size": len(CACHE)}}
'''

    # 중복 추가 방지: 아직 /embed_cached 가 없을 때만 파일에 덧붙입니다
    if "/embed_cached" not in Path("embedding_server.py").read_text(encoding="utf-8"):
        with open("embedding_server.py", "a", encoding="utf-8") as f:
            f.write(cache_patch)

    emb_proc.terminate(); emb_proc.wait(timeout=30)
    emb_proc = start_service("embedding_server", 8001, "embedding")

    repeated = ["재택근무 신청 방법"] * 20
    for endpoint in ["embed", "embed_cached"]:
        t0 = time.time()
        for q in repeated:
            requests.post(f"{EMB_URL}/{endpoint}", json={"texts": [q], "kind": "query"})
        print(f"{endpoint:<14} {(time.time()-t0)/len(repeated)*1000:>7.1f} ms/요청")

    m = requests.get(f"{EMB_URL}/metrics").json()
    hits, misses = m.get("cache_hits", 0), m.get("cache_misses", 0)
    ### START CODE HERE ### (hits 와 misses 로부터 캐시 적중률 hit_rate 를 계산하세요)
    hit_rate = hits / max(hits + misses, 1)
    ### END CODE HERE ###
    print(f"\n캐시 적중률 : {(hit_rate or 0):.1%}")

[embedding] 기동 완료 (20초) : {'status': 'ok', 'model': 'intfloat/multilingual-e5-small', 'dim': 384, 'device': 'cuda'}
embed             30.1 ms/요청
embed_cached       3.4 ms/요청

캐시 적중률 : 95.0%

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
CACHE_MAX_LIMIT = 10000
# ...
hit_rate = hits / max(hits + misses, 1)
```

반복 질의에서 지연이 크게 줄어듭니다. 상한이 없으면 메모리가 무한정 늘어나므로 LRU 방식으로 상한을 뒀습니다.

**사내 질문은 반복되므로** 이 캐시가 값집니다. "연차 며칠이야"를 100명이 묻습니다. 문서 인덱싱에서도 바뀌지 않은 문서를 다시 임베딩할 필요가 없어 유용합니다. 지난 노트의 파이프라인과 결합하면 증분 처리 효율이 더 오릅니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 3. 내부 서비스 API 키 인증 및 접근 통제

이론 노트에서 오케스트레이터만 사용자에게 노출하고 나머지는 내부 전용으로 두라고 했습니다. 내부 서비스를 보호하는 장치를 넣습니다.

In [ ]:
ensure_services()

# 실습 : 내부 서비스에 API 키 인증(/embed_secure)을 추가하고 검증하세요

### START CODE HERE ### (인증 실패 HTTP 상태 코드와 정상 호출용 헤더를 채우세요)
REJECT_STATUS_CODE = 401
VALID_AUTH_HEADERS = {"X-Internal-Key": "svc-key-abc123"}
### END CODE HERE ###

if REJECT_STATUS_CODE is None or VALID_AUTH_HEADERS is None:
    print("[안내] REJECT_STATUS_CODE 와 VALID_AUTH_HEADERS 빈칸을 채운 뒤 실행하세요.")
else:
    auth_patch = f'''
from fastapi import Depends, Header, HTTPException

INTERNAL_KEY = os.environ.get("INTERNAL_API_KEY", "")


def require_key(x_internal_key: str = Header(default="")):
    if not INTERNAL_KEY:
        return          # 키가 설정되지 않으면 검증 생략 (개발 모드)
    if x_internal_key != INTERNAL_KEY:
        raise HTTPException(status_code={REJECT_STATUS_CODE}, detail="internal key required")


@app.post("/embed_secure")
def embed_secure(req: EmbedRequest, _=Depends(require_key)):
    return embed(req)
'''

    if "/embed_secure" not in Path("embedding_server.py").read_text(encoding="utf-8"):
        with open("embedding_server.py", "a", encoding="utf-8") as f:
            f.write(auth_patch)

    emb_proc.terminate(); emb_proc.wait(timeout=30)
    emb_proc = start_service("embedding_server", 8001, "embedding",
                             env={"INTERNAL_API_KEY": "svc-key-abc123"})

    # 1) 키 없이 호출
    r = requests.post(f"{EMB_URL}/embed_secure", json={"texts": ["테스트"], "kind": "query"})
    print(f"키 없음 : {r.status_code} {r.json().get('detail', '')}")

    # 2) 틀린 키로 호출
    r = requests.post(f"{EMB_URL}/embed_secure", json={"texts": ["테스트"], "kind": "query"},
                      headers={"X-Internal-Key": "svc-key-wrong"})
    print(f"틀린 키 : {r.status_code}")

    # 3) 올바른 키로 호출
    r = requests.post(f"{EMB_URL}/embed_secure", json={"texts": ["테스트"], "kind": "query"},
                      headers=VALID_AUTH_HEADERS)
    print(f"올바른 키 : {r.status_code}, 벡터 {len(r.json()['vectors'])}개")

[embedding] 기동 완료 (16초) : {'status': 'ok', 'model': 'intfloat/multilingual-e5-small', 'dim': 384, 'device': 'cuda'}
키 없음 : 401 internal key required
틀린 키 : 401
올바른 키 : 200, 벡터 1개

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
REJECT_STATUS_CODE = 401
VALID_AUTH_HEADERS = {"X-Internal-Key": "svc-key-abc123"}
```

API 키는 방어의 한 겹일 뿐입니다. 실무에서는 세 겹을 함께 씁니다.

**네트워크 격리** : 내부 서비스를 외부에서 접근할 수 없는 대역에 두는 가장 강한 방어입니다. 실습에서 `--host 127.0.0.1`로 바인딩한 것이 이 역할입니다. `0.0.0.0`으로 바인딩하면 외부에서 접근할 수 있게 됩니다.

**인증** : 내부에서도 신원을 확인합니다. 사내망이 뚫렸을 때의 방어입니다.

**감사 로그** : 누가 무엇을 호출했는지 기록합니다. 사고 후 추적의 근거입니다.

보안 주간에서 이 세 겹을 다룹니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 4. 주기적 헬스체크와 지수 백오프 기반 자동 재기동(Supervisor)

Step 8에서 손으로 한 재기동을 자동화합니다. 서비스를 주기적으로 확인하다 중단되면 다시 기동하는 프로그램을 **슈퍼바이저**(supervisor)라고 합니다.

In [ ]:
ensure_services()

# 실습 : 서비스 상태를 주기적으로 감시하고 장애 시 지수 백오프로 자동 재기동하는 슈퍼바이저를 완성하세요

class ServiceSupervisor:
    def __init__(self, name, script, port, max_restarts=3):
        self.name, self.script, self.port = name, script, port
        self.max_restarts = max_restarts
        self.restarts = 0
        self.proc = None
        self.history = []
        self.running = False

    def health(self):
        try:
            r = requests.get(f"http://127.0.0.1:{self.port}/health", timeout=2)
            ### START CODE HERE ### (응답 상태 코드가 200인지 판별하세요)
            return (r.status_code == 200)
            ### END CODE HERE ###
        except requests.exceptions.RequestException:
            return False

    def start(self):
        log = open(f"{self.name}.log", "a")
        self.proc = subprocess.Popen(
            ["uvicorn", f"{self.script}:app", "--host", "127.0.0.1",
             "--port", str(self.port), "--log-level", "warning"],
            stdout=log, stderr=subprocess.STDOUT)
        self.history.append({"event": "start", "at": time.strftime("%H:%M:%S")})

    def watch(self, interval=3, fail_threshold=2):
        self.running = True
        fails = 0
        print(f"[{self.name}] 감시 시작")

        def loop():
            nonlocal fails
            while self.running:
                if self.health():
                    fails = 0
                else:
                    fails += 1
                    if fails >= fail_threshold:
                        if self.restarts >= self.max_restarts:
                            self.history.append({"event": "give_up",
                                                 "at": time.strftime("%H:%M:%S")})
                            print(f"[{self.name}] 재기동 한도 초과. 감시를 중단합니다")
                            self.running = False
                            break
                        self.restarts += 1
                        print(f"[{self.name}] 비정상 감지. 재기동 {self.restarts}회")
                        self.history.append({"event": "restart", "n": self.restarts,
                                             "at": time.strftime("%H:%M:%S")})
                        self.start()
                        ### START CODE HERE ### (지수 백오프: 2 ** self.restarts 와 30초 중 작은 값만큼 대기)
                        backoff_sec = min(2 ** self.restarts, 30)
                        ### END CODE HERE ###
                        time.sleep(backoff_sec or 2)
                        fails = 0
                time.sleep(interval)

        self._thread = threading.Thread(target=loop, daemon=True)
        self._thread.start()

    def stop(self):
        """감시를 멈춥니다. Step 10 정리 전에 반드시 호출하세요."""
        self.running = False
        if getattr(self, "_thread", None):
            self._thread.join(timeout=5)
        print(f"[{self.name}] 감시를 중단했습니다")


sup = ServiceSupervisor("llm", "llm_server", 8002)
if sup.health() is None:
    print("[안내] health() 와 backoff_sec 빈칸을 채운 뒤 실행하세요.")
else:
    sup.proc = llm_proc
    sup.watch()

    print("\nLLM 서버를 강제 종료합니다")
    llm_proc.terminate()
    time.sleep(45)          # 감지와 재기동 대기

    print(f"\n현재 상태 : {'정상' if sup.health() else '비정상'}")
    print(f"이력 : {sup.history}")
    llm_proc = sup.proc

[llm] 감시 시작

LLM 서버를 강제 종료합니다
[llm] 비정상 감지. 재기동 1회
[llm] 비정상 감지. 재기동 2회
[llm] 비정상 감지. 재기동 3회

현재 상태 : 정상
이력 : [{'event': 'restart', 'n': 1, 'at': '05:08:26'}, {'event': 'start', 'at': '05:08:26'}, {'event': 'restart', 'n': 2, 'at': '05:08:34'}, {'event': 'start', 'at': '05:08:34'}, {'event': 'restart', 'n': 3, 'at': '05:08:44'}, {'event': 'start', 'at': '05:08:44'}]

<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
# 1) health() 정상 응답 판별
return (r.status_code == 200)

# 2) 지수 백오프 대기 시간 계산
backoff_sec = min(2 ** self.restarts, 30)
```

핵심은 지수 백오프와 최대 재기동 횟수입니다. 자동 재기동을 하면 재기동으로 풀리는 장애의 복구 시간(MTTR, 교재 표 4-1)이 사람이 알아차리는 시간 대신 감지 주기와 기동 시간만큼으로 줄어듭니다.

**무한 재기동은 위험합니다.** 모델 파일이 손상되어 기동할 수 없는 상태라면 재기동을 반복하며 GPU 메모리를 계속 점유하고 로그를 끝없이 늘립니다. 한도를 두고 포기한 뒤 사람에게 알리는 것이 맞습니다.

실무에서는 이 역할을 systemd, supervisor, Kubernetes가 맡습니다. **무엇을 설정해야 하는지 알려고** 직접 만들어 봅니다. 재시도 한도와 백오프 설정의 의미를 모르면 도구를 잘못 설정합니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

#### 트랙 5. LLM 서버 다중 인스턴스 기동과 로드 밸런싱·장애 전환

이론 6.1절에서 병목이 되는 구성 요소만 늘린다고 했습니다. LLM 서버를 두 대로 늘립니다.

In [41]:
ensure_services()

# 실습 : LLM 서버를 두 대 기동하고 최소 부하(least-inflight) 분산과 장애 전환을 구현하세요

import itertools
from concurrent.futures import ThreadPoolExecutor

LLM_POOL = ["http://127.0.0.1:8002", "http://127.0.0.1:8003"]
_rr = itertools.cycle(LLM_POOL)
POOL_STATE = {u: {"healthy": True, "inflight": 0} for u in LLM_POOL}


def pick_llm():
    """healthy 인스턴스 중 진행 중 요청(inflight)이 가장 적은 것을 고릅니다"""
    ### START CODE HERE ### (정상 인스턴스 목록 추출 및 최소 inflight 인스턴스 반환 2줄)
    healthy = [u for u in LLM_POOL if POOL_STATE[u]["healthy"]]
    if not healthy:
        raise RuntimeError("사용 가능한 LLM 인스턴스가 없습니다")
    return min(healthy, key=lambda u: POOL_STATE[u]["inflight"])
    ### END CODE HERE ###


def call_llm_balanced(body, timeout=120):
    tried = set()
    while len(tried) < len(LLM_POOL):
        url = pick_llm()
        if url in tried:
            url = next(u for u in LLM_POOL if u not in tried)
        tried.add(url)
        POOL_STATE[url]["inflight"] += 1
        try:
            r = requests.post(f"{url}/v1/chat/completions", json=body, timeout=timeout)
            r.raise_for_status()
            POOL_STATE[url]["healthy"] = True
            return r.json(), url
        except Exception:
            POOL_STATE[url]["healthy"] = False
            print(f"  [{url}] 실패. 다른 인스턴스로 전환")
        finally:
            POOL_STATE[url]["inflight"] -= 1
    raise RuntimeError("모든 LLM 인스턴스 호출 실패")


def bench(pool_urls, n_workers=4, n_req=4):
    global LLM_POOL, POOL_STATE
    LLM_POOL = pool_urls
    POOL_STATE = {u: {"healthy": True, "inflight": 0} for u in LLM_POOL}

    body = {"messages": [{"role": "user", "content": "사내 규정의 목적을 한 문장으로."}],
            "max_tokens": 60}
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        list(ex.map(lambda _: call_llm_balanced(body), range(n_req)))
    total = time.time() - t0
    return n_req / total


try:
    test_pick = pick_llm()
except Exception:
    test_pick = None

if test_pick is None:
    print("[안내] pick_llm() 안의 healthy 와 return 빈칸을 채운 뒤 실행하세요.")
else:
    llm_proc2 = start_service("llm_server", 8003, "llm2")
    tp1 = bench(["http://127.0.0.1:8002"])
    tp2 = bench(["http://127.0.0.1:8002", "http://127.0.0.1:8003"])
    print(f"\n1대 : {tp1:.2f} req/s")
    print(f"2대 : {tp2:.2f} req/s  ({tp2/tp1:.2f}배)")

    LLM_POOL = ["http://127.0.0.1:8002", "http://127.0.0.1:8003"]
    POOL_STATE = {u: {"healthy": True, "inflight": 0} for u in LLM_POOL}
    llm_proc2.terminate(); llm_proc2.wait(timeout=30)
    print("\n2번 인스턴스 종료")
    _, used = call_llm_balanced({"messages": [{"role": "user", "content": "안녕"}],
                                "max_tokens": 20})
    print(f"요청이 처리된 인스턴스 : {used}")

[llm2] 기동 완료 (12초) : {'status': 'ok', 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}

1대 : 0.34 req/s
2대 : 0.72 req/s  (2.13배)

2번 인스턴스 종료
요청이 처리된 인스턴스 : http://127.0.0.1:8002


<details class="lms-answer">
<summary>풀이 보기</summary>

**핵심 정답 코드 (`### START CODE HERE ###` 영역)**

```python
healthy = [u for u in LLM_POOL if POOL_STATE[u]["healthy"]]
if not healthy:
    raise RuntimeError("사용 가능한 LLM 인스턴스가 없습니다")
return min(healthy, key=lambda u: POOL_STATE[u]["inflight"])
```

두 인스턴스가 **같은 GPU를 공유**해 GPU 연산은 여전히 직렬화되고 메모리만 두 배로 씁니다. 처리량이 두 배가 안 나오는 것이 정상입니다.

**같은 하드웨어에서 프로세스를 늘려도 처리량은 늘지 않습니다.** 실제 확장은 GPU를 늘리거나 한 GPU에서 배칭 효율을 올리는 것입니다. 비용이 훨씬 적게 드는 배칭 효율 올리기가 서빙 심화 주간의 주제입니다. 지금 만든 구조는 **GPU가 여러 장일 때** 의미가 있습니다.

다만 한 인스턴스가 멈춰도 서비스가 계속된다는 이중화의 이점은 유효합니다. 마지막 검증에서 확인했고 교재 표 4-1의 복제·고가용성에 해당합니다.

**실행 순서** : 실습 셀의 `### START CODE HERE ###` 블록 빈칸을 위 정답 코드로 채운 뒤 실행합니다. (전체 코드를 한 번에 덮어쓰려면 '전체 완성 코드 보기'를 펼쳐 복사하세요.)

</details>

## Step 10. 백그라운드 서비스 종료 및 리소스 정리

순서가 중요합니다. **트랙 4의 슈퍼바이저를 먼저 멈추고 서비스를 내립니다.** 반대로 하면 슈퍼바이저가 내려간 서버를 다시 기동합니다.

In [42]:
# 1) 트랙 4를 수행했다면 감시 스레드가 살아 있습니다. 서버를 내리기 전에 멈춥니다
sup = globals().get("sup")
if sup is not None:
    sup.stop()

# 2) 서비스 종료
for name, p in [("embedding", emb_proc), ("llm", llm_proc), ("qdrant", qdrant_proc)]:
    try:
        p.terminate()
        p.wait(timeout=20)
        print(f"[{name}] 종료")
    except Exception:
        try:
            p.kill()
        except Exception:
            pass

# 3) 트랙 4나 트랙 5가 새로 기동한 프로세스가 남아 있을 수 있습니다
for port in (8001, 8002, 8003):
    stop_port(port)

disable_guard()

[embedding] 종료
[llm] 종료
[qdrant] 종료
소켓 감시 해제


슈퍼바이저가 살아 있으면 **서버를 멈춰도 다시 기동되고** 좀비 프로세스가 GPU 메모리와 포트를 계속 점유합니다. 다음 실습에서 "왜 기동이 안 되지"로 나타나는데 원인이 몇 셀 앞에 있어 찾기 어렵습니다. 그래서 정리 셀을 이렇게까지 씁니다.

사내 운영에서도 인수인계 문서의 **재시작 절차에 순서와 그 이유를 한 줄씩** 적어 둡니다.

## Step 11. 실습 결과 점검과 정리

### Step 10. 결과 점검과 정리

(선택 대상: 트랙 1 동시 요청 부하 실험과 추론 잠금 Lock 제어 / 트랙 2 LRU 기반 임베딩 캐시 적용과 지연 시간 단축 / 트랙 3 내부 서비스 API 키 인증 및 접근 통제 / 트랙 4 주기적 헬스체크와 지수 백오프 기반 자동 재기동 Supervisor / 트랙 5 LLM 서버 다중 인스턴스 기동과 최소 부하 분산·장애 전환)
각자 수행한 두 트랙을 공유합니다. 아래 형식으로 정리해주세요.
1. **선택한 트랙과 이유**
2. **측정 결과** : 구간별 지연 시간(`embed_ms`, `search_ms`, `generate_ms`, `total_ms`)의 변화, 캐시 적중률 또는 다중 인스턴스 처리량(`RPS`), 장애 주입 시 폴백 전환 시간 및 자동 재기동 복구 시간(`MTTR`)
3. **폐쇄망 로컬 서빙의 한계를 만난 지점** : 단일 프로세스 추론 병목, 자식 프로세스(`Subprocess`)의 암묵적 외부 통신 시도, LLM 장애 시 원문 반환 폴백의 품질 차이, 캐시 메모리 상한 등 구체적으로
4. **이 아키텍처를 사내 폐쇄망에 배포한다면** : 반입 패키지·모델·컨테이너 검증 절차(`release_manifest.json`), 서버 프로세스 격리(`127.0.0.1`) 및 인증 구성, 허용 가능한 성능 저하 모드 수준

공유가 끝나면 오늘 만든 것을 한 문장으로 정리해봅시다.
우리는 외부와 단절된 환경에 패키지·모델·컨테이너를 해시 매니페스트로 반입하고, 내부 주소로 격리된 마이크로서비스 위에 자식 프로세스까지 감시하는 소켓 차단막과 2단계 폴백·자동 재기동 장치를 얹어 외부 통신 0건과 무중단 응답을 동시에 증명했습니다.
여기서 질문이 하나 생깁니다. 이 모든 것을 통틀어 부르는 이름이 있을까요? 단순히 인터넷을 끈 채 모델을 돌리는 오프라인 스크립트도 아니고 방화벽만 믿는 수동 운영도 아니고, 반입 무결성·통신 통제·장애 격리를 코드와 증거로 보증하는 이 시스템 아키텍처 전체를 말입니다.

<div style="font-size: 18px; line-height: 1.6;">

| 계층 | 오프라인/폐쇄망 위협 요인 | 구현한 방어 및 검증 장치 (`D44`) |
| :--- | :--- | :--- |
| **반입 계층 (Step 1)** | 하위 패키지 누락, 모델 심볼릭 링크 깨짐, 이미지 변조 | 전이 의존성 `.whl` + 로컬 모델 스냅샷 + 컨테이너 `@sha256` 통합 `release_manifest.json` |
| **네트워크·격리 계층 (Step 2~4)** | `0.0.0.0` 무인증 노출, 단일 프로세스 메모리 동반 추락 | `127.0.0.1` 로컬호스트 바인딩 격리 + 마이크로서비스 분리(`:8001`, `:8002`, `:6333`) |
| **보안 감시 계층 (Step 7)** | 라이브러리 암묵적 텔레메트리(Phone-Home) 및 자식 프로세스 우회 | `socket.connect`/`getaddrinfo` 후킹 + `sitecustomize.py`(`PYTHONPATH`) 전 프로세스 채증 |
| **가용성·관측 계층 (Step 6, 8)** | LLM/임베딩 서버 부분 장애 시 전체 `500 에러` 중단 | 구간별 지연 분해 + 2단계 폴백(`degraded_generate`/`search`) + 지수 백오프 슈퍼바이저 |

</div>

## 마무리 과제. 폐쇄망 RAG 시스템 구축 및 폐쇄 검증 보고서 작성

여러분의 문서로 에어갭 RAG를 구축하고 검증 보고서를 작성합니다.

**1단계. 반입물 준비**

1. 여러분 시스템에 필요한 패키지를 목록 파일 하나에 `==`로 적고 **한 번에** 번들을 받습니다.  
2. 잠금 파일을 만듭니다. 우리가 적은 줄 수와 실제로 담긴 개수를 함께 적습니다.  
3. 필요한 모델을 로컬에 다운로드하고 모델 레지스트리 파일을 작성합니다. 저장소 버전 번호를 함께 기록합니다.  
4. 모든 서비스가 이름이 아니라 경로로 로드하는지 확인합니다.

**2단계. 서비스 구축**

5. 임베딩 서버와 LLM 서버를 구현합니다. 각각 `/health`와 `/metrics`를 제공하고 요청 모델의 목록과 messages에 `Field(min_length=1)`, 각 텍스트에 `StringConstraints(max_length=...)` 같은 길이 제약을 넣어 Step 3-1의 API 테스트에서 잘못된 입력이 모두 4xx로 거절되어야 합니다(교재 6.3.3).  
6. 벡터DB를 서버 모드로 기동합니다. **설정 파일에서 텔레메트리가 꺼져 있는지 확인합니다.**  
7. 오케스트레이터를 구현합니다. 구간별 지연 계측을 포함해야 합니다.  
8. 모든 내부 서비스가 `127.0.0.1`에만 바인딩되어 있는지 확인합니다.

**3단계. 폐쇄 증명**

9. 소켓 감시를 활성화하고 전체 파이프라인을 실행합니다.  
10. 차단된 시도를 목록으로 정리합니다. 각 시도가 어느 라이브러리에서 왔고 왜 필요했는지 조사해 이론 6.3절의 세 유형(꺼도 되는 것 / 미리 넣어야 하는 것 / 대체해야 하는 것)으로 분류합니다.  
11. 차단 상태에서 파이프라인이 정상 동작함을 증명합니다. **여러분의 증명이 커버하지 못하는 범위를 함께 적습니다.** 감시가 닿지 않는 프로세스나 경로를 하나 이상 구체적으로 지목합니다.

**4단계. 장애 대응**

12. 성능 저하 모드를 최소 두 단계 구현합니다.  
13. 각 서비스를 하나씩 멈춰 폴백이 동작함을 확인합니다.  
14. 복구 후 정상 모드로 돌아오는지 확인합니다.

**5단계. 검증 보고서**

- **반입 명세서** : 대상 환경(운영체제, 파이썬 버전), 잠금 파일 이름, 모델 목록과 버전 번호, 총 크기  
- **아키텍처 다이어그램** : 세 서비스와 오케스트레이터, 통신 경로, 그리고 네트워크 경계 표시  
- **구간별 지연 표** : 각 구간의 평균과 비중  
- **외부 통신 차단 보고** : 차단된 시도 목록과 각각에 대한 조치, 그리고 증명이 닿지 않는 범위  
- **장애 대응 표** : 서비스별 장애 시 영향과 폴백 동작, 실제 검증 결과  
- **API 기반과의 비교** : 이론 노트의 표를 여러분 시스템 기준으로 채우기. 품질과 속도에서 무엇을 포기하고 무엇을 얻는지  
- **다음 개선 과제** : 측정 결과로 드러난 병목과 개선 방향

**평가 기준**

동작하는 시스템을 만들었는지가 아니라 **증명과 측정**으로 평가합니다. 폐쇄를 어떻게 증명했는지, 지연을 구간별로 쪼갰는지, 장애를 실제로 일으켜 폴백을 확인했는지를 봅니다. API 기반과의 비교에서는 **불리한 점을 먼저 적었는지**가 중요합니다. 프라이빗 AI 제안에서 그것이 신뢰의 기반입니다.